In [ ]:

import torch

print("PyTorch Version:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")


In [ ]:

from google.colab import drive

drive.mount('/content/drive')


In [ ]:

from pathlib import Path

# Main project folder
PROJECT_DIR = Path("/content/drive/MyDrive/Traffic_Parking_Project")

# Create project folders
folders = [
    "data/raw",
    "data/processed",
    "models",
    "results",
    "reports"
]

for folder in folders:
    (PROJECT_DIR / folder).mkdir(parents=True, exist_ok=True)

print("Project folders created successfully!")
print("Project location:", PROJECT_DIR)


In [ ]:

!kaggle --version


In [ ]:

import os

username = os.environ.get("KAGGLE_USERNAME")
key = os.environ.get("KAGGLE_KEY")

if username and key:
    print("Kaggle credentials are available!")
else:
    print("Kaggle credentials are not configured.")


In [ ]:

from google.colab import userdata
import os
import subprocess

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

result = subprocess.run(
    ["kaggle", "datasets", "list", "-s", "pklot"],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("Authentication successful!")
    print(result.stdout[:1500])
else:
    print("Authentication or dataset search failed:")
    print(result.stderr[:1000])


In [ ]:

from pathlib import Path
import subprocess

DATA_DIR = Path("/content/pklot_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

result = subprocess.run([
    "kaggle", "datasets", "download",
    "-d", "ammarnassanalhajali/pklot-dataset",
    "-p", str(DATA_DIR)
])

if result.returncode == 0:
    print("Dataset downloaded successfully!")
else:
    print("Download failed!")


In [ ]:

for file in DATA_DIR.iterdir():
    print(file.name, "-", round(file.stat().st_size / (1024**2), 2), "MB")


In [ ]:

import zipfile
from collections import Counter

zip_path = "/content/pklot_data/pklot-dataset.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    files = z.namelist()

    print("Total files:", len(files))

    print("\nFirst 20 files:")
    for name in files[:20]:
        print(name)

    extensions = Counter(
        name.split(".")[-1].lower()
        for name in files
        if "." in name and not name.endswith("/")
    )

    print("\nFile types:")
    print(extensions)


In [ ]:

import zipfile

zip_path = "/content/pklot_data/pklot-dataset.zip"

with zipfile.ZipFile(zip_path, "r") as z:

    json_files = [
        name for name in z.namelist()
        if name.endswith(".json")
    ]

    print("JSON files:")
    for name in json_files:
        print(name)

    print("\nREADME content:")
    if "README.dataset.txt" in z.namelist():
        content = z.read("README.dataset.txt").decode(
            "utf-8", errors="replace"
        )
        print(content[:2000])

    print("\nFirst JSON preview:")
    if json_files:
        content = z.read(json_files[0]).decode(
            "utf-8", errors="replace"
        )
        print(content[:2500])


In [ ]:

import zipfile
import json
from collections import Counter

zip_path = "/content/pklot_data/pklot-dataset.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    for split in ["train", "valid", "test"]:
        path = f"{split}/_annotations.coco.json"
        data = json.loads(z.read(path))

        categories = {
            c["id"]: c["name"]
            for c in data["categories"]
        }

        counts = Counter(
            a["category_id"]
            for a in data["annotations"]
        )

        print(f"\n--- {split.upper()} ---")
        print("Images:", len(data["images"]))
        print("Annotations:", len(data["annotations"]))

        for category_id, name in categories.items():
            print(name, ":", counts[category_id])


In [ ]:

import zipfile
import json
from collections import defaultdict

zip_path = "/content/pklot_data/pklot-dataset.zip"
split_dates = {}

with zipfile.ZipFile(zip_path, "r") as z:
    for split in ["train", "valid", "test"]:
        data = json.loads(
            z.read(f"{split}/_annotations.coco.json")
        )

        dates = {
            img["file_name"][:10]
            for img in data["images"]
        }

        split_dates[split] = dates

        print(f"{split}: {len(dates)} unique dates")
        print("Examples:", sorted(dates)[:5])

print("\n--- Date overlap ---")

pairs = [
    ("train", "valid"),
    ("train", "test"),
    ("valid", "test")
]

for a, b in pairs:
    overlap = split_dates[a] & split_dates[b]
    print(f"{a} vs {b}: {len(overlap)} shared dates")
    print("Examples:", sorted(overlap)[:5])


In [ ]:

import zipfile
import json
from collections import Counter

zip_path = "/content/pklot_data/pklot-dataset.zip"

date_counts = Counter()
all_images = []

with zipfile.ZipFile(zip_path, "r") as z:
    for split in ["train", "valid", "test"]:
        data = json.loads(
            z.read(f"{split}/_annotations.coco.json")
        )

        for img in data["images"]:
            filename = img["file_name"]
            date = filename[:10]

            date_counts[date] += 1
            all_images.append((split, filename, date))

print("Total images:", len(all_images))
print("Unique dates:", len(date_counts))

print("\nTop 10 dates by image count:")
for date, count in date_counts.most_common(10):
    print(date, ":", count)

print("\nDates with fewest images:")
for date, count in sorted(
    date_counts.items(),
    key=lambda x: x[1]
)[:10]:
    print(date, ":", count)


In [ ]:

import zipfile
import json

zip_path = "/content/pklot_data/pklot-dataset.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    data = json.loads(
        z.read("train/_annotations.coco.json")
    )

    print("Image fields:")
    print(list(data["images"][0].keys()))

    print("\nAnnotation fields:")
    print(list(data["annotations"][0].keys()))

    print("\nSample annotation:")
    print(data["annotations"][0])

    print("\nDataset metadata:")
    print(data.get("info", {}))


In [ ]:

import subprocess

result = subprocess.run(
    ["kaggle", "datasets", "list", "-s", "PKLot"],
    capture_output=True,
    text=True
)

print(result.stdout[:5000])



In [ ]:

import subprocess

result = subprocess.run(
    [
        "kaggle", "datasets", "files",
        "omarbourouba/pklotpatches"
    ],
    capture_output=True,
    text=True
)

print("Exit code:", result.returncode)
print(result.stdout[:6000])

if result.returncode != 0:
    print(result.stderr[:1500])


In [ ]:

import subprocess

result = subprocess.run(
    [
        "kaggle", "datasets", "files",
        "blanderbuss/parking-lot-dataset"
    ],
    capture_output=True,
    text=True
)

print("Exit code:", result.returncode)
print(result.stdout[:7000])

if result.returncode != 0:
    print(result.stderr[:1500])


In [ ]:

import shutil

total, used, free = shutil.disk_usage("/content")

print(f"Total space: {total / (1024**3):.2f} GB")
print(f"Used space: {used / (1024**3):.2f} GB")
print(f"Free space: {free / (1024**3):.2f} GB")



In [ ]:

from pathlib import Path
import subprocess

original_dir = Path("/content/pklot_original")
original_dir.mkdir(parents=True, exist_ok=True)

result = subprocess.run([
    "kaggle", "datasets", "download",
    "-d", "blanderbuss/parking-lot-dataset",
    "-p", str(original_dir)
])

if result.returncode == 0:
    print("Download completed successfully!")
else:
    print("Download failed!")



In [ ]:

for file in original_dir.iterdir():
    print(
        file.name,
        "-",
        round(file.stat().st_size / (1024**3), 2),
        "GB"
    )


In [ ]:

import zipfile
from collections import Counter

zip_path = "/content/pklot_original/parking-lot-dataset.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    files = z.namelist()

    images = [f for f in files if f.lower().endswith(".jpg")]
    xml_files = [f for f in files if f.lower().endswith(".xml")]

    locations = Counter()
    weather = Counter()

    for path in images:
        parts = path.split("/")

        for location in ["PUCPR", "UFPR04", "UFPR05"]:
            if location in parts:
                locations[location] += 1

        for condition in ["Sunny", "Cloudy", "Rainy"]:
            if condition in parts:
                weather[condition] += 1

    print("Total files:", len(files))
    print("JPG images:", len(images))
    print("XML annotations:", len(xml_files))

    print("\nImages by location:")
    print(dict(locations))

    print("\nImages by weather:")
    print(dict(weather))

    print("\nFirst 5 files:")
    for name in files[:5]:
        print(name)


In [ ]:

import zipfile
from collections import Counter

zip_path = "/content/pklot_original/parking-lot-dataset.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    files = z.namelist()

    # توزيع الصور حسب المجلدات الرئيسية
    folders = Counter()

    for name in files:
        if name.lower().endswith(".jpg"):
            parts = name.split("/")
            folder = "/".join(parts[:4])
            folders[folder] += 1

    print("TOP IMAGE FOLDERS:")
    for folder, count in folders.most_common(15):
        print(folder, ":", count)

    # قراءة أول ملف XML
    xml_files = [
        name for name in files
        if name.lower().endswith(".xml")
    ]

    print("\nFIRST XML PATH:")
    print(xml_files[0])

    print("\nXML CONTENT PREVIEW:")
    content = z.read(xml_files[0]).decode(
        "utf-8", errors="replace"
    )
    print(content[:3500])


In [ ]:

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

project = Path("/content/drive/MyDrive/Traffic_Parking_Project")

print("Project exists:", project.exists())

if project.exists():
    print("\nFiles in project:")
    for item in project.rglob("*"):
        if item.is_file() and item.suffix.lower() == ".zip":
            print(item, round(item.stat().st_size / 1024**3, 2), "GB")


In [ ]:

from google.colab import userdata
from pathlib import Path
import subprocess
import os

# Restore Kaggle credentials
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

# Permanent storage in Google Drive
save_dir = Path(
    "/content/drive/MyDrive/Traffic_Parking_Project/data/raw"
)

save_dir.mkdir(parents=True, exist_ok=True)

# Download PKLot
result = subprocess.run([
    "kaggle", "datasets", "download",
    "-d", "blanderbuss/parking-lot-dataset",
    "-p", str(save_dir)
])

if result.returncode == 0:
    print("Download completed successfully!")
else:
    print("Download failed!")


In [ ]:

zip_path = save_dir / "parking-lot-dataset.zip"

if zip_path.exists():
    print("File saved successfully!")
    print("Size:", round(zip_path.stat().st_size / 1024**3, 2), "GB")
else:
    print("File not found.")


In [ ]:

import zipfile
from collections import Counter

zip_path = (
    "/content/drive/MyDrive/Traffic_Parking_Project/"
    "data/raw/parking-lot-dataset.zip"
)

with zipfile.ZipFile(zip_path, "r") as z:
    segmented = [
        name for name in z.namelist()
        if "pklotsegmented" in name.lower()
        and name.lower().endswith(".jpg")
    ]

    print("Total segmented images:", len(segmented))

    print("\nFirst 10 paths:")
    for name in segmented[:10]:
        print(name)

    labels = Counter()

    for name in segmented:
        parts = name.lower().split("/")

        if "empty" in parts:
            labels["Empty"] += 1
        elif "occupied" in parts:
            labels["Occupied"] += 1
        else:
            labels["Other"] += 1

    print("\nClassification labels:")
    print(dict(labels))


In [ ]:

import zipfile
import pandas as pd
from pathlib import Path

zip_path = (
    "/content/drive/MyDrive/Traffic_Parking_Project/"
    "data/raw/parking-lot-dataset.zip"
)

records = []

with zipfile.ZipFile(zip_path, "r") as z:
    for name in z.namelist():
        if "pklotsegmented" not in name.lower():
            continue
        if not name.lower().endswith(".jpg"):
            continue

        parts = name.split("/")
        lower = [p.lower() for p in parts]

        idx = lower.index("pklotsegmented")
        fields = parts[idx + 1:]

        if len(fields) < 5:
            continue

        location, weather, date, label = fields[:4]

        if label.lower() not in ["empty", "occupied"]:
            continue

        records.append({
            "image_path": name,
            "location": location,
            "weather": weather,
            "date": date,
            "label": label.lower()
        })

df = pd.DataFrame(records)

output_path = (
    "/content/drive/MyDrive/Traffic_Parking_Project/"
    "data/processed/pklot_index.csv"
)

Path(output_path).parent.mkdir(
    parents=True, exist_ok=True
)

df.to_csv(output_path, index=False)

print("Total indexed images:", len(df))

print("\nLocations:")
print(df["location"].value_counts())

print("\nLabels:")
print(df["label"].value_counts())

print("\nUnique dates by location:")
print(df.groupby("location")["date"].nunique())

print("\nCSV saved to:", output_path)


In [ ]:

import pandas as pd

index_path = (
    "/content/drive/MyDrive/Traffic_Parking_Project/"
    "data/processed/pklot_index.csv"
)

df = pd.read_csv(index_path)

daily_stats = (
    df.groupby(["location", "date", "label"])
      .size()
      .unstack(fill_value=0)
      .reset_index()
)

daily_stats["total"] = (
    daily_stats["empty"] + daily_stats["occupied"]
)

print("Daily statistics by location:")
print(
    daily_stats.groupby("location")[
        ["empty", "occupied", "total"]
    ].agg(["min", "max", "mean"]).round(1)
)

print("\nPUC first 10 days:")
print(
    daily_stats[daily_stats["location"] == "PUC"]
    .sort_values("date")
    .head(10)
    .to_string(index=False)
)


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

base = Path(
    "/content/drive/MyDrive/Traffic_Parking_Project"
)

df = pd.read_csv(base / "data/processed/pklot_index.csv")

# Separate locations
puc = df[df["location"] == "PUC"].copy()
external_test = df[
    df["location"].isin(["UFPR04", "UFPR05"])
].copy()

# Count images per day
day_counts = puc.groupby("date").size()

# Randomize whole days reproducibly
rng = np.random.default_rng(42)
days = rng.permutation(day_counts.index.to_numpy())

target_val = int(len(puc) * 0.20)

val_days = []
val_count = 0

for day in days:
    if val_count >= target_val:
        break
    val_days.append(day)
    val_count += day_counts[day]

# Split by day
train_df = puc[~puc["date"].isin(val_days)].copy()
val_df = puc[puc["date"].isin(val_days)].copy()
test_df = external_test.copy()

# Save splits
output_dir = base / "data/processed/splits"
output_dir.mkdir(parents=True, exist_ok=True)

train_df.to_csv(output_dir / "train.csv", index=False)
val_df.to_csv(output_dir / "val.csv", index=False)
test_df.to_csv(output_dir / "test.csv", index=False)

print("TRAIN:", len(train_df))
print("VALIDATION:", len(val_df))
print("TEST:", len(test_df))

print("\nTrain days:", train_df["date"].nunique())
print("Validation days:", val_df["date"].nunique())

print("\nTrain/Validation day overlap:",
      len(set(train_df["date"]) & set(val_df["date"])))

print("\nLabel distribution:")
for name, subset in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(name, subset["label"].value_counts().to_dict())

print("\nSaved to:", output_dir)


In [ ]:

import pandas as pd
from pathlib import Path

base = Path(
    "/content/drive/MyDrive/Traffic_Parking_Project"
)

split_dir = base / "data/processed/splits"

train = pd.read_csv(split_dir / "train.csv")
val = pd.read_csv(split_dir / "val.csv")
test = pd.read_csv(split_dir / "test.csv")

def sample_grouped(df, total_per_class, seed=42):
    selected = []

    for label in ["empty", "occupied"]:
        subset = df[df["label"] == label]

        # Spread samples across location and date groups
        groups = [
            group.sample(frac=1, random_state=seed)
            for _, group in subset.groupby(
                ["location", "date"], sort=True
            )
        ]

        # Round-robin selection across groups
        rows = []
        position = 0

        while len(rows) < total_per_class:
            added = False

            for group in groups:
                if position < len(group):
                    rows.append(group.iloc[position])
                    added = True

                    if len(rows) == total_per_class:
                        break

            if not added:
                break

            position += 1

        selected.extend(rows)

    return pd.DataFrame(selected).sample(
        frac=1, random_state=seed
    ).reset_index(drop=True)

train_small = sample_grouped(train, 10000)
val_small = sample_grouped(val, 2000)
test_small = sample_grouped(test, 2000)

sample_dir = base / "data/processed/samples"
sample_dir.mkdir(parents=True, exist_ok=True)

train_small.to_csv(sample_dir / "train_sample.csv", index=False)
val_small.to_csv(sample_dir / "val_sample.csv", index=False)
test_small.to_csv(sample_dir / "test_sample.csv", index=False)

for name, subset in [
    ("Train", train_small),
    ("Validation", val_small),
    ("Test", test_small)
]:
    print(f"\n{name}: {len(subset)} images")
    print(subset["label"].value_counts().to_dict())
    print("Locations:", subset["location"].unique())
    print("Days:", subset["date"].nunique())

print("\nSamples saved to:", sample_dir)


In [ ]:

import zipfile
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import shutil

base = Path(
    "/content/drive/MyDrive/Traffic_Parking_Project"
)

zip_path = base / "data/raw/parking-lot-dataset.zip"
sample_dir = base / "data/processed/samples"
output_dir = base / "data/processed/images"

splits = {
    "train": "train_sample.csv",
    "val": "val_sample.csv",
    "test": "test_sample.csv"
}

with zipfile.ZipFile(zip_path, "r") as z:

    for split, csv_file in splits.items():
        df = pd.read_csv(sample_dir / csv_file)

        print(f"\nExtracting {split}: {len(df)} images")

        for _, row in tqdm(
            df.iterrows(),
            total=len(df)
        ):
            source_path = row["image_path"]
            label = row["label"]

            # Preserve original filename
            filename = Path(source_path).name

            # Avoid collisions across different dates/locations
            safe_name = (
                f"{row['location']}_"
                f"{row['date']}_"
                f"{filename}"
            )

            target_dir = output_dir / split / label
            target_dir.mkdir(parents=True, exist_ok=True)

            target_path = target_dir / safe_name

            if target_path.exists():
                continue

            with z.open(source_path) as src:
                with open(target_path, "wb") as dst:
                    shutil.copyfileobj(src, dst)

        print(f"{split} extraction completed!")

print("\nAll selected images extracted successfully!")


In [ ]:

from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import random

base = Path(
    "/content/drive/MyDrive/Traffic_Parking_Project"
)

image_dir = base / "data/processed/images"

# 1. Count extracted images
for split in ["train", "val", "test"]:
    print(f"\n{split.upper()}")

    for label in ["empty", "occupied"]:
        folder = image_dir / split / label
        count = len(list(folder.glob("*.jpg")))
        print(f"{label}: {count}")

# 2. Display sample images
random.seed(42)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))

for row, label in enumerate(["empty", "occupied"]):
    folder = image_dir / "train" / label
    files = random.sample(list(folder.glob("*.jpg")), 5)

    for col, file in enumerate(files):
        with Image.open(file) as img:
            axes[row, col].imshow(img.convert("RGB"))

        axes[row, col].set_title(label.capitalize())
        axes[row, col].axis("off")

plt.suptitle("PKLot Sample Images", fontsize=15)
plt.tight_layout()
plt.show()


In [ ]:

import pandas as pd
from pathlib import Path

base = Path(
    "/content/drive/MyDrive/Traffic_Parking_Project"
)

sample_dir = base / "data/processed/samples"

for split in ["train", "val", "test"]:

    df = pd.read_csv(
        sample_dir / f"{split}_sample.csv"
    )

    df["target_name"] = (
        df["location"].astype(str) + "_" +
        df["date"].astype(str) + "_" +
        df["image_path"].apply(
            lambda x: Path(x).name
        )
    )

    # Duplicates within each label folder
    duplicates = df.duplicated(
        subset=["label", "target_name"],
        keep=False
    )

    unique_targets = df.drop_duplicates(
        subset=["label", "target_name"]
    )

    print(f"\n{split.upper()}")
    print("CSV images:", len(df))
    print("Unique target paths:", len(unique_targets))
    print("Duplicate rows:", duplicates.sum())
    print("Expected collisions:",
          len(df) - len(unique_targets))

    if duplicates.any():
        print("\nExamples:")
        print(
            df.loc[
                duplicates,
                ["image_path", "label", "target_name"]
            ].head(6).to_string(index=False)
        )


In [ ]:

import zipfile
import pandas as pd
from pathlib import Path

base = Path("/content/drive/MyDrive/Traffic_Parking_Project")
zip_path = base / "data/raw/parking-lot-dataset.zip"
sample_dir = base / "data/processed/samples"

frames = []

for split in ["train", "val", "test"]:
    df = pd.read_csv(sample_dir / f"{split}_sample.csv")
    df["split"] = split
    frames.append(df)

samples = pd.concat(frames, ignore_index=True)

# Create the set ONCE
needed_paths = set(samples["image_path"])

with zipfile.ZipFile(zip_path, "r") as z:
    info_map = {
        info.filename: (info.CRC, info.file_size)
        for info in z.infolist()
        if info.filename in needed_paths
    }

samples["fingerprint"] = samples["image_path"].map(
    lambda p: info_map[p]
)

for split, df in samples.groupby("split"):
    print(f"\n{split.upper()}")
    print("Total:", len(df))
    print("Unique fingerprints:", df["fingerprint"].nunique())
    print("Likely duplicates:",
          len(df) - df["fingerprint"].nunique())

cross_split = samples.groupby("fingerprint")["split"].nunique()

print("\nFingerprints shared across splits:",
      (cross_split > 1).sum())


In [ ]:

import zipfile
import pandas as pd
from pathlib import Path

base = Path("/content/drive/MyDrive/Traffic_Parking_Project")
sample_dir = base / "data/processed/samples"
zip_path = base / "data/raw/parking-lot-dataset.zip"

frames = []

for split in ["train", "val", "test"]:
    df = pd.read_csv(sample_dir / f"{split}_sample.csv")
    df["split"] = split
    frames.append(df)

samples = pd.concat(frames, ignore_index=True)

with zipfile.ZipFile(zip_path) as z:
    fingerprints = {
        p: (z.getinfo(p).CRC, z.getinfo(p).file_size)
        for p in set(samples["image_path"])
    }

samples["fingerprint"] = samples["image_path"].map(fingerprints)

# Check whether identical fingerprints have conflicting labels
conflicts = (
    samples.groupby("fingerprint")["label"]
    .nunique()
)

print("Conflicting labels:", (conflicts > 1).sum())

# Check duplicates under different ZIP paths
duplicates = samples[
    samples.duplicated("fingerprint", keep=False)
].copy()

print("Rows belonging to duplicate groups:", len(duplicates))

print("\nExample duplicate pairs:")
for fingerprint, group in list(
    duplicates.groupby("fingerprint")
)[:3]:
    print("\n---")
    print(group[["split", "label", "image_path"]].to_string(index=False))


In [ ]:

import zipfile
import pandas as pd
from pathlib import Path

base = Path("/content/drive/MyDrive/Traffic_Parking_Project")
zip_path = base / "data/raw/parking-lot-dataset.zip"
split_dir = base / "data/processed/splits"
sample_dir = base / "data/processed/samples"

targets = {
    "train": 10000,
    "val": 2000,
    "test": 2000
}

# Load original splits and selected samples
data = {}

for split in targets:
    full = pd.read_csv(split_dir / f"{split}.csv")
    sample = pd.read_csv(sample_dir / f"{split}_sample.csv")
    data[split] = (full, sample)

# Read ZIP metadata once
with zipfile.ZipFile(zip_path) as z:
    info = {
        x.filename: (x.CRC, x.file_size)
        for x in z.infolist()
        if x.filename.lower().endswith(".jpg")
    }

# Track unique images across all splits
used = set()
final_samples = {}

for split in ["train", "val", "test"]:
    full, sample = data[split]

    sample = sample.copy()
    sample["fingerprint"] = sample["image_path"].map(info)

    # Keep one copy of each image
    sample = sample.drop_duplicates(
        "fingerprint"
    ).copy()

    selected = [sample]

    for label in ["empty", "occupied"]:
        current = sample[sample["label"] == label]
        needed = targets[split] - len(current)

        candidates = full[
            (full["label"] == label) &
            (~full["image_path"].isin(sample["image_path"]))
        ].sample(frac=1, random_state=42).copy()

        candidates["fingerprint"] = (
            candidates["image_path"].map(info)
        )

        candidates = candidates.drop_duplicates(
            "fingerprint"
        )

        candidates = candidates[
            ~candidates["fingerprint"].isin(
                used | set(sample["fingerprint"])
            )
        ]

        additions = candidates.head(needed)

        if len(additions) != needed:
            raise ValueError(
                f"{split}/{label}: insufficient unique images"
            )

        selected.append(additions)
        used.update(additions["fingerprint"])

    final = pd.concat(selected, ignore_index=True)

    used.update(final["fingerprint"])

    if final["fingerprint"].duplicated().any():
        raise ValueError(f"Duplicates found in {split}")

    final = final.drop(columns=["fingerprint"])

    output = sample_dir / f"{split}_clean.csv"
    final.to_csv(output, index=False)

    final_samples[split] = final

    print(f"\n{split.upper()}")
    print("Total:", len(final))
    print(final["label"].value_counts().to_dict())
    print("Saved:", output)

print("\nClean sample CSV files created.")


In [ ]:

import zipfile
import pandas as pd
import shutil
import hashlib
from pathlib import Path
from tqdm.auto import tqdm

base = Path("/content/drive/MyDrive/Traffic_Parking_Project")
zip_path = base / "data/raw/parking-lot-dataset.zip"
sample_dir = base / "data/processed/samples"
old_dir = base / "data/processed/images"
new_dir = base / "data/processed/images_clean"

splits = {"train": 20000, "val": 4000, "test": 4000}

with zipfile.ZipFile(zip_path, "r") as z:
    for split, expected in splits.items():
        df = pd.read_csv(sample_dir / f"{split}_clean.csv")
        assert len(df) == expected

        print(f"\nProcessing {split}...")

        for row in tqdm(df.itertuples(index=False), total=len(df)):
            source = row.image_path
            label = row.label

            # Unique name based on complete ZIP path
            uid = hashlib.sha256(source.encode()).hexdigest()[:20]
            filename = f"{uid}.jpg"

            destination = new_dir / split / label / filename
            destination.parent.mkdir(parents=True, exist_ok=True)

            if destination.exists():
                continue

            # Old extraction filename
            old_name = (
                f"{row.location}_{row.date}_"
                f"{Path(source).name}"
            )
            old_path = old_dir / split / label / old_name

            # Reuse only when CRC and size match
            source_info = z.getinfo(source)

            if old_path.exists():
                import zlib

                crc = 0
                size = 0

                with open(old_path, "rb") as f:
                    while chunk := f.read(1024 * 1024):
                        crc = zlib.crc32(chunk, crc)
                        size += len(chunk)

                if (
                    crc == source_info.CRC
                    and size == source_info.file_size
                ):
                    shutil.copy2(old_path, destination)
                    continue

            # Extract from ZIP
            with z.open(source) as src:
                with open(destination, "wb") as dst:
                    shutil.copyfileobj(src, dst)

        print(f"\n{split.upper()} verification:")

        total = 0
        for label in ["empty", "occupied"]:
            count = len(list((new_dir / split / label).glob("*.jpg")))
            print(f"{label}: {count}")
            total += count

        print("TOTAL:", total)
        assert total == expected, f"Unexpected count in {split}"

print("\nAll 28,000 clean images are ready!")


In [ ]:

from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import random

base = Path("/content/drive/MyDrive/Traffic_Parking_Project")
image_dir = base / "data/processed/images_clean"

random.seed(42)

fig, axes = plt.subplots(3, 6, figsize=(15, 9))

for row, split in enumerate(["train", "val", "test"]):
    for label, start_col in [("empty", 0), ("occupied", 3)]:
        folder = image_dir / split / label
        files = random.sample(list(folder.glob("*.jpg")), 3)

        for j, file in enumerate(files):
            with Image.open(file) as img:
                axes[row, start_col + j].imshow(
                    img.convert("RGB")
                )

            axes[row, start_col + j].set_title(
                f"{split.upper()} - {label}"
            )
            axes[row, start_col + j].axis("off")

plt.suptitle("PKLot Dataset - Visual Inspection", fontsize=16)
plt.tight_layout()
plt.show()

# Check image dimensions and readability
for split in ["train", "val", "test"]:
    for label in ["empty", "occupied"]:
        folder = image_dir / split / label
        file = next(folder.glob("*.jpg"))

        with Image.open(file) as img:
            img.verify()

        with Image.open(file) as img:
            print(
                split, label,
                "size:", img.size,
                "mode:", img.mode
            )


In [ ]:

import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from pathlib import Path

base = Path("/content/drive/MyDrive/Traffic_Parking_Project")
image_dir = base / "data/processed/images_clean"

# Select GPU if available
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

# Image preprocessing
train_transform = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Load datasets
train_dataset = datasets.ImageFolder(
    image_dir / "train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    image_dir / "val",
    transform=eval_transform
)

test_dataset = datasets.ImageFolder(
    image_dir / "test",
    transform=eval_transform
)

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

print("Classes:", train_dataset.classes)
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

# Verify one batch
images, labels = next(iter(train_loader))

print("Batch images shape:", images.shape)
print("Batch labels shape:", labels.shape)
print("Image dtype:", images.dtype)


In [ ]:

import torch
import torch.nn as nn

class ParkingCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Create model from scratch
model = ParkingCNN().to(device)

# Count trainable parameters
total_params = sum(
    p.numel() for p in model.parameters()
)

print(model)
print("\nTotal parameters:", total_params)

# Test one batch
images, labels = next(iter(train_loader))
images = images.to(device)

with torch.no_grad():
    outputs = model(images)

print("Input shape:", images.shape)
print("Output shape:", outputs.shape)


In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

# Reproducibility
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Start with a fresh model
model = ParkingCNN().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 5
best_val_loss = float("inf")
history = []

base = Path("/content/drive/MyDrive/Traffic_Parking_Project")
model_dir = base / "models"
result_dir = base / "results"

model_dir.mkdir(parents=True, exist_ok=True)
result_dir.mkdir(parents=True, exist_ok=True)

best_model_path = model_dir / "parking_cnn_best.pth"

for epoch in range(epochs):

    # TRAINING
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{epochs} - Train"
    ):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        train_correct += (
            outputs.argmax(dim=1) == labels
        ).sum().item()
        train_total += labels.size(0)

    train_loss /= train_total
    train_acc = 100 * train_correct / train_total

    # VALIDATION
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in tqdm(
            val_loader,
            desc=f"Epoch {epoch+1}/{epochs} - Val"
        ):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            val_correct += (
                outputs.argmax(dim=1) == labels
            ).sum().item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = 100 * val_correct / val_total

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc
    })

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": val_loss,
            "val_accuracy": val_acc,
            "classes": train_dataset.classes
        }, best_model_path)

        print("Best model saved!")

    # Save results after every epoch
    pd.DataFrame(history).to_csv(
        result_dir / "cnn_training_history.csv",
        index=False
    )

    print(
        f"\nEpoch {epoch+1}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%\n"
    )

print("\nTraining completed!")
print("Best model:", best_model_path)


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

base = Path("/content/drive/MyDrive/Traffic_Parking_Project")
results_dir = base / "results"

history = pd.read_csv(
    results_dir / "cnn_training_history.csv"
)

# Accuracy curve
plt.figure(figsize=(8, 5))
plt.plot(
    history["epoch"],
    history["train_accuracy"],
    marker="o",
    label="Train Accuracy"
)
plt.plot(
    history["epoch"],
    history["val_accuracy"],
    marker="o",
    label="Validation Accuracy"
)
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("CNN Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.savefig(
    results_dir / "cnn_accuracy_curve.png",
    dpi=300
)
plt.show()

# Loss curve
plt.figure(figsize=(8, 5))
plt.plot(
    history["epoch"],
    history["train_loss"],
    marker="o",
    label="Train Loss"
)
plt.plot(
    history["epoch"],
    history["val_loss"],
    marker="o",
    label="Validation Loss"
)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("CNN Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.savefig(
    results_dir / "cnn_loss_curve.png",
    dpi=300
)
plt.show()

print("Training plots saved successfully!")


In [ ]:

import torch
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score
)
from torch.utils.data import DataLoader

base = Path("/content/drive/MyDrive/Traffic_Parking_Project")

# Load best checkpoint
checkpoint = torch.load(
    base / "models/parking_cnn_best.pth",
    map_location=device,
    weights_only=True
)

model = ParkingCNN().to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Best epoch:", checkpoint["epoch"])

# Avoid multiprocessing warnings
safe_val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

all_true = []
all_pred = []

with torch.no_grad():
    for images, labels in safe_val_loader:
        images = images.to(device)
        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        all_true.extend(labels.tolist())
        all_pred.extend(predictions.cpu().tolist())

classes = val_dataset.classes

print("\nValidation Accuracy:",
      round(accuracy_score(all_true, all_pred) * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(
    all_true,
    all_pred,
    target_names=classes,
    digits=4
))

cm = confusion_matrix(all_true, all_pred)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=classes
).plot(cmap="Blues", values_format="d")

plt.title("CNN Validation Confusion Matrix")
plt.tight_layout()
plt.savefig(
    base / "results/cnn_validation_confusion_matrix.png",
    dpi=300
)
plt.show()

report = classification_report(
    all_true,
    all_pred,
    target_names=classes,
    output_dict=True
)

pd.DataFrame(report).transpose().to_csv(
    base / "results/cnn_validation_metrics.csv"
)

print("\nValidation metrics saved successfully!")


In [ ]:

import torch
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score
)

base = Path("/content/drive/MyDrive/Traffic_Parking_Project")

# Load best CNN checkpoint
checkpoint = torch.load(
    base / "models/parking_cnn_best.pth",
    map_location=device,
    weights_only=True
)

model = ParkingCNN().to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

safe_test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

all_true = []
all_pred = []

with torch.no_grad():
    for images, labels in safe_test_loader:
        images = images.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        all_true.extend(labels.tolist())
        all_pred.extend(predictions.cpu().tolist())

classes = test_dataset.classes

print("TEST ACCURACY:",
      round(accuracy_score(all_true, all_pred) * 100, 2), "%")

print("\nTEST CLASSIFICATION REPORT:")
print(classification_report(
    all_true,
    all_pred,
    target_names=classes,
    digits=4
))

cm = confusion_matrix(all_true, all_pred)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=classes
).plot(cmap="Blues", values_format="d")

plt.title("CNN Cross-Location Test Confusion Matrix")
plt.tight_layout()

plt.savefig(
    base / "results/cnn_test_confusion_matrix.png",
    dpi=300
)
plt.show()

report = classification_report(
    all_true,
    all_pred,
    target_names=classes,
    output_dict=True
)

pd.DataFrame(report).transpose().to_csv(
    base / "results/cnn_test_metrics.csv"
)

print("\nCross-location test completed!")


In [ ]:

import pandas as pd
from pathlib import Path

base = Path("/content/drive/MyDrive/Traffic_Parking_Project")
results_dir = base / "results"

validation = pd.read_csv(
    results_dir / "cnn_validation_metrics.csv",
    index_col=0
)

test = pd.read_csv(
    results_dir / "cnn_test_metrics.csv",
    index_col=0
)

summary = pd.DataFrame([
    {
        "Model": "CNN From Scratch",
        "Dataset": "PKLot",
        "Evaluation": "Validation (PUC)",
        "Accuracy": validation.loc["accuracy", "f1-score"],
        "Macro Precision": validation.loc["macro avg", "precision"],
        "Macro Recall": validation.loc["macro avg", "recall"],
        "Macro F1": validation.loc["macro avg", "f1-score"]
    },
    {
        "Model": "CNN From Scratch",
        "Dataset": "PKLot",
        "Evaluation": "Cross-location Test",
        "Accuracy": test.loc["accuracy", "f1-score"],
        "Macro Precision": test.loc["macro avg", "precision"],
        "Macro Recall": test.loc["macro avg", "recall"],
        "Macro F1": test.loc["macro avg", "f1-score"]
    }
])

output = results_dir / "model_comparison.csv"
summary.to_csv(output, index=False)

print(summary.round(4).to_string(index=False))
print("\nSaved:", output)


In [ ]:

!pip -q install wandb


In [ ]:

import wandb

wandb.login()


In [ ]:

import wandb
import pandas as pd
from pathlib import Path

base = Path("/content/drive/MyDrive/Traffic_Parking_Project")

history = pd.read_csv(
    base / "results/cnn_training_history.csv"
)

run = wandb.init(
    project="traffic-parking-occupancy",
    name="cnn-from-scratch-baseline",
    job_type="historical-import",
    config={
        "model": "ParkingCNN",
        "dataset": "PKLot",
        "training": "from_scratch",
        "train_location": "PUC",
        "test_locations": ["UFPR04", "UFPR05"],
        "train_samples": 20000,
        "validation_samples": 4000,
        "test_samples": 4000,
        "image_size": 96,
        "batch_size": 32,
        "epochs": 5,
        "optimizer": "Adam",
        "learning_rate": 0.001,
        "parameters": 93954,
        "seed": 42
    }
)

# Upload previous training history
for _, row in history.iterrows():
    wandb.log({
        "train/loss": float(row["train_loss"]),
        "train/accuracy": float(row["train_accuracy"]),
        "validation/loss": float(row["val_loss"]),
        "validation/accuracy": float(row["val_accuracy"])
    }, step=int(row["epoch"]))

# Final results
wandb.summary["best_epoch"] = 4
wandb.summary["validation_accuracy"] = 0.9945
wandb.summary["cross_location_test_accuracy"] = 0.9310
wandb.summary["cross_location_test_macro_f1"] = 0.9308

wandb.finish()

print("W&B experiment uploaded successfully!")


In [ ]:

import torch
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

base = Path("/content/drive/MyDrive/Traffic_Parking_Project")

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Not available")
print("Project folder exists:", base.exists())
print("CNN checkpoint exists:", (base / "models/parking_cnn_best.pth").exists())
print("Train images exist:", (base / "data/processed/images_clean/train").exists())


In [ ]:

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Image preprocessing
train_transform = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Dataset paths
images_dir = base / "data/processed/images_clean"

train_dataset = datasets.ImageFolder(
    images_dir / "train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    images_dir / "val",
    transform=eval_transform
)

test_dataset = datasets.ImageFolder(
    images_dir / "test",
    transform=eval_transform
)

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

print("Train images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Test images:", len(test_dataset))
print("Classes:", train_dataset.classes)

images, labels = next(iter(train_loader))
print("Batch shape:", images.shape)
print("Labels shape:", labels.shape)


In [ ]:

import torch
import torch.nn as nn
from torchvision import models
from torchvision.models import ResNet18_Weights

# Load pretrained ResNet18
weights = ResNet18_Weights.DEFAULT
resnet18 = models.resnet18(weights=weights)

# Freeze pretrained layers
for param in resnet18.parameters():
    param.requires_grad = False

# Replace classification layer
num_features = resnet18.fc.in_features
resnet18.fc = nn.Linear(num_features, 2)

# Move model to GPU
resnet18 = resnet18.to(device)

# Count parameters
total_params = sum(
    p.numel() for p in resnet18.parameters()
)

trainable_params = sum(
    p.numel()
    for p in resnet18.parameters()
    if p.requires_grad
)

print("Model: ResNet18")
print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
print("Frozen parameters:",
      total_params - trainable_params)

# Check output shape
resnet18.eval()

dummy_input = torch.randn(2, 3, 96, 96).to(device)

with torch.no_grad():
    output = resnet18(dummy_input)

print("Output shape:", output.shape)


In [ ]:

import time
import copy
import pandas as pd
import wandb
import torch.optim as optim

# Reproducibility
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# Training setup
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    resnet18.fc.parameters(),
    lr=0.001
)

epochs = 5
best_val_loss = float("inf")
best_epoch = 0
history = []

# Start W&B tracking
run = wandb.init(
    project="traffic-parking-occupancy",
    name="resnet18-frozen-backbone",
    job_type="training",
    config={
        "model": "ResNet18",
        "pretrained": "ImageNet",
        "training": "frozen_backbone",
        "dataset": "PKLot",
        "train_location": "PUC",
        "train_samples": 20000,
        "validation_samples": 4000,
        "image_size": 96,
        "batch_size": 32,
        "epochs": epochs,
        "optimizer": "Adam",
        "learning_rate": 0.001,
        "seed": 42
    }
)

for epoch in range(epochs):
    start_time = time.time()

    # Train classification head
    resnet18.eval()
    resnet18.fc.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = resnet18(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        train_correct += (
            outputs.argmax(1) == labels
        ).sum().item()
        train_total += labels.size(0)

    train_loss /= train_total
    train_acc = 100 * train_correct / train_total

    # Validation
    resnet18.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = resnet18(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            val_correct += (
                outputs.argmax(1) == labels
            ).sum().item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = 100 * val_correct / val_total

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1

        torch.save({
            "model_state_dict": resnet18.state_dict(),
            "epoch": best_epoch,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
            "classes": train_dataset.classes
        }, base / "models/resnet18_frozen_best.pth")

    # Log metrics
    wandb.log({
        "epoch": epoch + 1,
        "train/loss": train_loss,
        "train/accuracy": train_acc,
        "validation/loss": val_loss,
        "validation/accuracy": val_acc
    })

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc
    })

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Acc: {val_acc:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Time: {time.time()-start_time:.1f}s",
        flush=True
    )

# Save training history
pd.DataFrame(history).to_csv(
    base / "results/resnet18_frozen_history.csv",
    index=False
)

wandb.summary["best_epoch"] = best_epoch
wandb.summary["best_validation_loss"] = best_val_loss
wandb.finish()

print("\nTraining completed!")
print("Best epoch:", best_epoch)
print("Best validation loss:", round(best_val_loss, 4))


In [ ]:

import shutil
import time
from pathlib import Path

source = base / "data/processed/images_clean"
destination = Path("/content/parking_images")

start = time.time()

for split in ["train", "val", "test"]:
    src = source / split
    dst = destination / split

    print(f"Copying {split}...", flush=True)

    shutil.copytree(
        src,
        dst,
        dirs_exist_ok=True
    )

    count = sum(1 for p in dst.rglob("*.jpg"))

    print(f"{split}: {count} images copied", flush=True)

print(
    f"Total copying time: {(time.time()-start)/60:.2f} minutes"
)


In [ ]:

import time
from torchvision import datasets
from torch.utils.data import DataLoader

# Use local Colab storage
local_images = "/content/parking_images"

train_dataset = datasets.ImageFolder(
    f"{local_images}/train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    f"{local_images}/val",
    transform=eval_transform
)

test_dataset = datasets.ImageFolder(
    f"{local_images}/test",
    transform=eval_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Test loading speed
start = time.time()

for i, (images, labels) in enumerate(train_loader):
    if i == 19:
        break

elapsed = time.time() - start

print("Train images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Test images:", len(test_dataset))
print("20 batches loading time:", round(elapsed, 2), "seconds")
print("Average per batch:", round(elapsed / 20, 3), "seconds")


In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torchvision.models import ResNet18_Weights

# Set random seed
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# Create a fresh pretrained model
resnet18 = models.resnet18(
    weights=ResNet18_Weights.DEFAULT
)

# Freeze backbone
for param in resnet18.parameters():
    param.requires_grad = False

# Replace classification layer
resnet18.fc = nn.Linear(
    resnet18.fc.in_features, 2
)

resnet18 = resnet18.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    resnet18.fc.parameters(),
    lr=0.001
)

print("Model reset successfully")
print("Device:", device)
print("Trainable parameters:",
      sum(p.numel() for p in resnet18.parameters()
          if p.requires_grad))


In [ ]:

import time
import pandas as pd
import wandb

epochs = 5
best_val_loss = float("inf")
best_epoch = 0
history = []

run = wandb.init(
    project="traffic-parking-occupancy",
    name="resnet18-frozen-local",
    job_type="training",
    config={
        "model": "ResNet18",
        "pretrained": "ImageNet",
        "training": "frozen_backbone",
        "dataset": "PKLot",
        "train_samples": 20000,
        "validation_samples": 4000,
        "image_size": 96,
        "batch_size": 32,
        "epochs": 5,
        "optimizer": "Adam",
        "learning_rate": 0.001,
        "seed": 42
    }
)

for epoch in range(epochs):
    start_time = time.time()

    # Keep backbone in evaluation mode
    resnet18.eval()
    resnet18.fc.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = resnet18(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        train_correct += (
            outputs.argmax(1) == labels
        ).sum().item()
        train_total += labels.size(0)

    train_loss /= train_total
    train_acc = 100 * train_correct / train_total

    # Validation
    resnet18.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = resnet18(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            val_correct += (
                outputs.argmax(1) == labels
            ).sum().item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = 100 * val_correct / val_total

    # Save best model to Google Drive
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1

        torch.save({
            "model_state_dict": resnet18.state_dict(),
            "epoch": best_epoch,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
            "classes": train_dataset.classes
        }, base / "models/resnet18_frozen_best.pth")

    wandb.log({
        "epoch": epoch + 1,
        "train/loss": train_loss,
        "train/accuracy": train_acc,
        "validation/loss": val_loss,
        "validation/accuracy": val_acc
    })

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc
    })

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Acc: {val_acc:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Time: {time.time()-start_time:.1f}s",
        flush=True
    )

pd.DataFrame(history).to_csv(
    base / "results/resnet18_frozen_history.csv",
    index=False
)

wandb.summary["best_epoch"] = best_epoch
wandb.summary["best_validation_loss"] = best_val_loss
wandb.finish()

print("\nTraining completed!")
print("Best epoch:", best_epoch)
print("Best validation loss:", round(best_val_loss, 4))


In [ ]:

import pandas as pd

history = pd.read_csv(
    base / "results/resnet18_frozen_history.csv"
)

print("ResNet18 Frozen Training Results")
print("-" * 55)

print(history.to_string(index=False))

best_row = history.loc[
    history["val_loss"].idxmin()
]

print("\nBest Model Results:")
print("Best Epoch:", int(best_row["epoch"]))
print("Validation Accuracy:",
      round(best_row["val_accuracy"], 2), "%")
print("Validation Loss:",
      round(best_row["val_loss"], 4))


In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torchvision.models import ResNet18_Weights

# Reproducibility
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# Load a fresh pretrained ResNet18
resnet18_full = models.resnet18(
    weights=ResNet18_Weights.DEFAULT
)

# Replace classification layer
resnet18_full.fc = nn.Linear(
    resnet18_full.fc.in_features,
    2
)

# Enable training for all layers
for param in resnet18_full.parameters():
    param.requires_grad = True

resnet18_full = resnet18_full.to(device)

# Loss and optimizer
criterion_full = nn.CrossEntropyLoss()

optimizer_full = optim.Adam(
    resnet18_full.parameters(),
    lr=0.0001
)

total_params = sum(
    p.numel() for p in resnet18_full.parameters()
)

trainable_params = sum(
    p.numel()
    for p in resnet18_full.parameters()
    if p.requires_grad
)

print("Model: ResNet18 Full Fine-tuning")
print("Device:", device)
print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
print("All layers trainable:",
      total_params == trainable_params)


In [ ]:

import time
import pandas as pd
import wandb
import torch

epochs = 5
best_val_loss = float("inf")
best_epoch = 0
history_full = []

run = wandb.init(
    project="traffic-parking-occupancy",
    name="resnet18-full-finetuning",
    job_type="training",
    config={
        "model": "ResNet18",
        "pretrained": "ImageNet",
        "training": "full_finetuning",
        "dataset": "PKLot",
        "train_location": "PUC",
        "train_samples": 20000,
        "validation_samples": 4000,
        "image_size": 96,
        "batch_size": 32,
        "epochs": epochs,
        "optimizer": "Adam",
        "learning_rate": 0.0001,
        "seed": 42
    }
)

for epoch in range(epochs):
    start_time = time.time()

    # Train all layers
    resnet18_full.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer_full.zero_grad()

        outputs = resnet18_full(images)
        loss = criterion_full(outputs, labels)

        loss.backward()
        optimizer_full.step()

        train_loss += loss.item() * images.size(0)
        train_correct += (
            outputs.argmax(1) == labels
        ).sum().item()
        train_total += labels.size(0)

    train_loss /= train_total
    train_acc = 100 * train_correct / train_total

    # Validation
    resnet18_full.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = resnet18_full(images)
            loss = criterion_full(outputs, labels)

            val_loss += loss.item() * images.size(0)
            val_correct += (
                outputs.argmax(1) == labels
            ).sum().item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = 100 * val_correct / val_total

    # Save best checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1

        torch.save({
            "model_state_dict": resnet18_full.state_dict(),
            "epoch": best_epoch,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
            "classes": train_dataset.classes
        }, base / "models/resnet18_full_best.pth")

    # W&B tracking
    wandb.log({
        "epoch": epoch + 1,
        "train/loss": train_loss,
        "train/accuracy": train_acc,
        "validation/loss": val_loss,
        "validation/accuracy": val_acc
    })

    history_full.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc
    })

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Acc: {val_acc:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Time: {time.time()-start_time:.1f}s",
        flush=True
    )

# Save history
pd.DataFrame(history_full).to_csv(
    base / "results/resnet18_full_history.csv",
    index=False
)

wandb.summary["best_epoch"] = best_epoch
wandb.summary["best_validation_loss"] = best_val_loss

wandb.finish()

print("\nFull Fine-tuning completed!")
print("Best epoch:", best_epoch)
print("Best validation loss:", round(best_val_loss, 4))


In [ ]:

import pandas as pd

history_full = pd.read_csv(
    base / "results/resnet18_full_history.csv"
)

print("ResNet18 Full Fine-tuning Results")
print("-" * 60)
print(history_full.to_string(index=False))

best_row = history_full.loc[
    history_full["val_loss"].idxmin()
]

print("\nBest Model Results:")
print("Best Epoch:", int(best_row["epoch"]))
print(
    "Validation Accuracy:",
    round(best_row["val_accuracy"], 2), "%"
)
print(
    "Validation Loss:",
    round(best_row["val_loss"], 4)
)


In [ ]:

import torch
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

def evaluate_resnet(model, checkpoint_path, model_name):
    # Load the best saved checkpoint
    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=True
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()

    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device, non_blocking=True)
            outputs = model(images)
            predictions = outputs.argmax(dim=1)

            all_labels.extend(labels.numpy().tolist())
            all_predictions.extend(predictions.cpu().numpy().tolist())

    accuracy = accuracy_score(all_labels, all_predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    print(f"\n{model_name}")
    print("Accuracy:", round(accuracy * 100, 2), "%")
    print("Macro Precision:", round(precision, 4))
    print("Macro Recall:", round(recall, 4))
    print("Macro F1:", round(f1, 4))
    print("\nClassification Report:")
    print(classification_report(
        all_labels,
        all_predictions,
        target_names=test_dataset.classes
    ))

    cm = confusion_matrix(all_labels, all_predictions)
    display = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=test_dataset.classes
    )
    display.plot(cmap="Blues", values_format="d")
    plt.title(f"{model_name} - Cross-location Test")
    plt.savefig(
        base / f"results/{model_name}_test_confusion_matrix.png",
        dpi=150,
        bbox_inches="tight"
    )
    plt.show()

    return {
        "model": model_name,
        "test_accuracy": accuracy,
        "test_macro_precision": precision,
        "test_macro_recall": recall,
        "test_macro_f1": f1
    }

# Frozen Backbone
frozen_results = evaluate_resnet(
    resnet18,
    base / "models/resnet18_frozen_best.pth",
    "resnet18_frozen"
)

# Full Fine-tuning
full_results = evaluate_resnet(
    resnet18_full,
    base / "models/resnet18_full_best.pth",
    "resnet18_full"
)

comparison = pd.DataFrame([
    {
        "model": "cnn_from_scratch",
        "test_accuracy": 0.9310,
        "test_macro_precision": 0.9361,
        "test_macro_recall": 0.9310,
        "test_macro_f1": 0.9308
    },
    frozen_results,
    full_results
])

comparison.to_csv(
    base / "results/resnet18_test_comparison.csv",
    index=False
)

print("\nFinal Cross-location Comparison:")
print(comparison.to_string(index=False))


In [ ]:

import pandas as pd

final_comparison = pd.DataFrame([
    {
        "model": "CNN From Scratch",
        "validation_accuracy": 99.45,
        "test_accuracy": 93.10,
        "test_macro_f1": 0.9308
    },
    {
        "model": "ResNet18 Frozen",
        "validation_accuracy": 99.80,
        "test_accuracy": 96.125,
        "test_macro_f1": 0.96124
    },
    {
        "model": "ResNet18 Full Fine-tuning",
        "validation_accuracy": 99.90,
        "test_accuracy": 99.05,
        "test_macro_f1": 0.99050
    }
])

output_path = base / "results/final_model_comparison.csv"

final_comparison.to_csv(
    output_path,
    index=False
)

print(final_comparison.to_string(index=False))
print("\nSaved to:", output_path)


In [ ]:

from pathlib import Path

raw_dir = base / "data/raw"

print("Files and folders in data/raw:")
for item in raw_dir.iterdir():
    print("-", item.name)

print("\nCNRPark-related files:")
matches = [
    str(p) for p in raw_dir.rglob("*")
    if "cnr" in p.name.lower()
]

if matches:
    for item in matches[:20]:
        print(item)
else:
    print("No CNRPark files found.")


In [ ]:

import requests

url = "http://claudiotest.isti.cnr.it/park-datasets/CNR-EXT/"

try:
    response = requests.get(
        url,
        timeout=20,
        allow_redirects=True
    )

    print("Status code:", response.status_code)
    print("Final URL:", response.url)
    print("Page preview:")
    print(response.text[:500])

except Exception as e:
    print("Connection error:", str(e))



In [ ]:

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

url = "http://claudiotest.isti.cnr.it/park-datasets/CNR-EXT/"

response = requests.get(url, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

print("CNRPark+EXT Directory Contents:\n")

for link in soup.find_all("a", href=True):
    name = link.get_text(" ", strip=True)
    href = link["href"]

    if href not in ("../", "/", ""):
        print("Name:", name)
        print("URL:", urljoin(url, href))
        print("-" * 50)


In [ ]:

from pathlib import Path
import requests

url = (
    "http://claudiotest.isti.cnr.it/"
    "park-datasets/CNR-EXT/"
    "CNR-EXT-Patches-150x150.zip"
)

destination = (
    base / "data/raw/CNR-EXT-Patches-150x150.zip"
)

destination.parent.mkdir(parents=True, exist_ok=True)

# Check file size before downloading
response = requests.head(
    url,
    allow_redirects=True,
    timeout=30
)

size_bytes = int(
    response.headers.get("Content-Length", 0)
)

print("HTTP Status:", response.status_code)

if size_bytes:
    print(
        "File size:",
        round(size_bytes / (1024**3), 2),
        "GB"
    )
else:
    print("File size: Not provided by server")

print("Destination:", destination)
print("Already downloaded:", destination.exists())


In [ ]:

import requests
from pathlib import Path
from tqdm.auto import tqdm

url = (
    "http://claudiotest.isti.cnr.it/"
    "park-datasets/CNR-EXT/"
    "CNR-EXT-Patches-150x150.zip"
)

destination = base / "data/raw/CNR-EXT-Patches-150x150.zip"
temporary = destination.with_suffix(".zip.part")

destination.parent.mkdir(parents=True, exist_ok=True)

if destination.exists():
    print("File already exists:", destination)
else:
    downloaded = temporary.stat().st_size if temporary.exists() else 0

    headers = {"Range": f"bytes={downloaded}-"} if downloaded else {}

    with requests.get(
        url,
        headers=headers,
        stream=True,
        timeout=(30, 120)
    ) as response:
        response.raise_for_status()

        if downloaded and response.status_code != 206:
            downloaded = 0
            mode = "wb"
        else:
            mode = "ab" if downloaded else "wb"

        total = int(response.headers.get("Content-Length", 0))

        with open(temporary, mode) as file, tqdm(
            total=total,
            unit="B",
            unit_scale=True,
            desc="Downloading CNRPark+EXT"
        ) as progress:

            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)
                    progress.update(len(chunk))

    temporary.rename(destination)

print("\nDownload completed.")
print("File:", destination)
print("Size (GB):", round(destination.stat().st_size / 1024**3, 3))


In [ ]:

import zipfile
from collections import Counter

zip_path = base / "data/raw/CNR-EXT-Patches-150x150.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    files = [f for f in z.infolist() if not f.is_dir()]

    print("ZIP structure:", "OK")
    print("Total files:", len(files))

    print("\nFirst 20 files:")
    for f in files[:20]:
        print(f.filename)

    print("\nTop-level folders:")
    folders = Counter(
        f.filename.split("/")[0]
        for f in files
    )
    for name, count in folders.most_common(15):
        print(name, ":", count)

    print("\nChecking ZIP integrity...")
    bad_file = z.testzip()

    if bad_file is None:
        print("ZIP integrity: PASSED")
    else:
        print("ZIP integrity: FAILED")
        print("Corrupted file:", bad_file)


In [ ]:

import zipfile

zip_path = base / "data/raw/CNR-EXT-Patches-150x150.zip"

with zipfile.ZipFile(zip_path, "r") as z:

    label_files = sorted([
        name for name in z.namelist()
        if name.startswith("LABELS/")
        and not name.endswith("/")
    ])

    print("Total label files:", len(label_files))

    print("\nLabel file names:")
    for name in label_files:
        print("-", name)

    if label_files:
        first_label = label_files[0]

        print("\n" + "=" * 60)
        print("Preview of:", first_label)
        print("=" * 60)

        content = z.read(first_label).decode(
            "utf-8",
            errors="replace"
        )

        print(content[:2000])


In [ ]:

import zipfile
from collections import Counter

zip_path = base / "data/raw/CNR-EXT-Patches-150x150.zip"

label_names = [
    "all", "train", "val", "test",
    "sunny", "rainy", "overcast"
]

with zipfile.ZipFile(zip_path, "r") as z:
    for name in label_names:
        content = z.read(f"LABELS/{name}.txt").decode(
            "utf-8", errors="replace"
        )

        rows = [
            line.strip().split()
            for line in content.splitlines()
            if line.strip()
        ]

        labels = Counter(
            row[-1] for row in rows if len(row) >= 2
        )

        print(f"{name}:")
        print("  Total:", len(rows))
        print("  Labels:", dict(labels))

        if name == "test":
            print("  First 3 examples:")
            for row in rows[:3]:
                print("   ", row)

        print("-" * 45)


In [ ]:

import zipfile
import random
import io
import matplotlib.pyplot as plt
from PIL import Image

zip_path = base / "data/raw/CNR-EXT-Patches-150x150.zip"

with zipfile.ZipFile(zip_path, "r") as z:

    lines = z.read("LABELS/test.txt").decode(
        "utf-8"
    ).splitlines()

    samples = {"0": [], "1": []}

    for line in lines:
        parts = line.strip().split()

        if len(parts) == 2:
            image_path, label = parts

            if label in samples:
                samples[label].append(image_path)

    random.seed(42)

    fig, axes = plt.subplots(2, 3, figsize=(12, 8))

    for row, label in enumerate(["0", "1"]):

        selected = random.sample(samples[label], 3)

        for col, image_path in enumerate(selected):

            full_path = "PATCHES/" + image_path

            image_bytes = z.read(full_path)
            image = Image.open(
                io.BytesIO(image_bytes)
            ).convert("RGB")

            axes[row, col].imshow(image)
            axes[row, col].set_title(
                f"Label = {label}",
                fontsize=13
            )
            axes[row, col].axis("off")

    plt.suptitle(
        "CNRPark+EXT - Label Verification",
        fontsize=16
    )
    plt.tight_layout()
    plt.show()


In [ ]:

import zipfile
import random
import shutil
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

zip_path = base / "data/raw/CNR-EXT-Patches-150x150.zip"

local_dir = Path("/content/cnrpark_test")
index_path = base / "data/processed/cnrpark_test_sample.csv"

random.seed(42)

with zipfile.ZipFile(zip_path, "r") as z:
    lines = z.read("LABELS/test.txt").decode("utf-8").splitlines()

    records = []

    for line in lines:
        parts = line.strip().split()

        if len(parts) == 2:
            image_path, label = parts

            records.append({
                "image_path": image_path,
                "label": int(label),
                "weather": image_path.split("/")[0],
                "date": image_path.split("/")[1],
                "camera": image_path.split("/")[2]
            })

    df = pd.DataFrame(records)

    sample = (
        df.groupby("label", group_keys=False)
          .sample(n=2000, random_state=42)
          .reset_index(drop=True)
    )

    sample["class_name"] = sample["label"].map({
        0: "empty",
        1: "occupied"
    })

    index_path.parent.mkdir(parents=True, exist_ok=True)
    sample.to_csv(index_path, index=False)

    for row in tqdm(
        sample.itertuples(index=False),
        total=len(sample),
        desc="Extracting CNRPark sample"
    ):
        source = "PATCHES/" + row.image_path

        destination = (
            local_dir /
            row.class_name /
            Path(row.image_path).name
        )

        destination.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        with z.open(source) as src, open(destination, "wb") as dst:
            shutil.copyfileobj(src, dst)

print("\nExtraction completed.")
print("Total selected:", len(sample))
print("Class distribution:")
print(sample["class_name"].value_counts())
print("\nCameras:", sample["camera"].nunique())
print("Dates:", sample["date"].nunique())
print("Weather conditions:", sample["weather"].nunique())
print("Index saved:", index_path)


In [ ]:

import torch
from pathlib import Path
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

cnr_dir = Path("/content/cnrpark_test")

cnr_transform = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

cnr_test_dataset = datasets.ImageFolder(
    root=cnr_dir,
    transform=cnr_transform
)

cnr_test_loader = DataLoader(
    cnr_test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

print("Dataset size:", len(cnr_test_dataset))
print("Classes:", cnr_test_dataset.classes)
print("Class mapping:", cnr_test_dataset.class_to_idx)
print("Number of batches:", len(cnr_test_loader))

images, labels = next(iter(cnr_test_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

assert len(cnr_test_dataset) == 4000
assert cnr_test_dataset.class_to_idx == {
    "empty": 0,
    "occupied": 1
}

print("\nCNRPark DataLoader: READY")


In [ ]:

import torch
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    ConfusionMatrixDisplay
)

def evaluate_cnr(model, checkpoint_path, model_name):
    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=True
    )

    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()

    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for images, labels in cnr_test_loader:
            images = images.to(device, non_blocking=True)

            outputs = model(images)
            predictions = outputs.argmax(dim=1)

            all_labels.extend(labels.numpy().tolist())
            all_predictions.extend(
                predictions.cpu().numpy().tolist()
            )

    accuracy = accuracy_score(
        all_labels, all_predictions
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            all_labels,
            all_predictions,
            average="macro",
            zero_division=0
        )
    )

    cm = confusion_matrix(
        all_labels,
        all_predictions,
        labels=[0, 1]
    )

    print(f"\n{model_name}")
    print("Accuracy:", round(accuracy * 100, 2), "%")
    print("Macro Precision:", round(precision, 4))
    print("Macro Recall:", round(recall, 4))
    print("Macro F1:", round(f1, 4))
    print("Confusion Matrix:\n", cm)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["empty", "occupied"]
    )

    disp.plot(cmap="Blues", values_format="d")
    plt.title(f"{model_name} - CNRPark+EXT")

    plt.savefig(
        base / f"results/{model_name}_cnr_confusion_matrix.png",
        dpi=150,
        bbox_inches="tight"
    )
    plt.show()

    return {
        "model": model_name,
        "cnr_accuracy": accuracy,
        "cnr_macro_precision": precision,
        "cnr_macro_recall": recall,
        "cnr_macro_f1": f1
    }


cnr_results = []

cnr_results.append(
    evaluate_cnr(
        cnn_model,
        base / "models/parking_cnn_best.pth",
        "cnn_from_scratch"
    )
)

cnr_results.append(
    evaluate_cnr(
        resnet18,
        base / "models/resnet18_frozen_best.pth",
        "resnet18_frozen"
    )
)

cnr_results.append(
    evaluate_cnr(
        resnet18_full,
        base / "models/resnet18_full_best.pth",
        "resnet18_full"
    )
)

cnr_comparison = pd.DataFrame(cnr_results)

cnr_comparison.to_csv(
    base / "results/cnrpark_external_comparison.csv",
    index=False
)

print("\nCNRPark+EXT Comparison:")
print(cnr_comparison.to_string(index=False))


In [ ]:

import torch
import torch.nn as nn

# Same CNN architecture used during training
class ParkingCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
    nn.Flatten(),
    nn.Dropout(0.3),
    nn.Identity(),
    nn.Linear(128, 2)
)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x


cnn_model = ParkingCNN().to(device)

checkpoint_path = base / "models/parking_cnn_best.pth"

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=True
)

cnn_model.load_state_dict(
    checkpoint["model_state_dict"]
)

cnn_model.eval()

print("CNN restored successfully!")
print(
    "Total parameters:",
    sum(p.numel() for p in cnn_model.parameters())
)
print("Checkpoint:", checkpoint_path)


In [ ]:

import torch

checkpoint = torch.load(
    base / "models/parking_cnn_best.pth",
    map_location="cpu",
    weights_only=True
)

state = checkpoint["model_state_dict"]

print("Classifier layers in checkpoint:")

for key, value in state.items():
    if key.startswith("classifier"):
        print(key, tuple(value.shape))

print("\nCheckpoint fields:")
print(list(checkpoint.keys()))


In [ ]:

import torch

saved_state = checkpoint["model_state_dict"]
current_state = cnn_model.state_dict()

print("Saved layers:", len(saved_state))
print("Current layers:", len(current_state))

missing = set(saved_state) - set(current_state)
extra = set(current_state) - set(saved_state)

shape_mismatch = [
    key for key in saved_state
    if key in current_state
    and saved_state[key].shape != current_state[key].shape
]

print("Missing layers:", missing)
print("Extra layers:", extra)
print("Shape mismatches:", shape_mismatch)

print("\nCurrent classifier:")
print(cnn_model.classifier)

print("\nModel structure:")
print(cnn_model)


In [ ]:

import torch
import torch.nn as nn

# Restore the original classifier architecture
cnn_model.pool = nn.Identity()

cnn_model.classifier = nn.Sequential(
    nn.AdaptiveAvgPool2d((1, 1)),
    nn.Flatten(),
    nn.Dropout(0.3),
    nn.Linear(128, 2)
)

# Reload the original trained weights
checkpoint = torch.load(
    base / "models/parking_cnn_best.pth",
    map_location=device,
    weights_only=True
)

cnn_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

cnn_model = cnn_model.to(device)
cnn_model.eval()

print("CNN architecture corrected successfully!")
print("Classifier:")
print(cnn_model.classifier)
print(
    "Total parameters:",
    sum(p.numel() for p in cnn_model.parameters())
)


In [ ]:

# Evaluate CNN on the external CNRPark+EXT sample

cnn_cnr_result = evaluate_cnr(
    cnn_model,
    base / "models/parking_cnn_best.pth",
    "cnn_from_scratch"
)

print("\nCNN External Evaluation Completed")
print("Accuracy:", round(cnn_cnr_result["cnr_accuracy"] * 100, 2), "%")
print("Macro F1:", round(cnn_cnr_result["cnr_macro_f1"], 4))


In [ ]:

frozen_cnr_result = evaluate_cnr(
    resnet18,
    base / "models/resnet18_frozen_best.pth",
    "resnet18_frozen"
)

print("\nResNet18 Frozen External Evaluation Completed")
print(
    "Accuracy:",
    round(frozen_cnr_result["cnr_accuracy"] * 100, 2),
    "%"
)
print(
    "Macro F1:",
    round(frozen_cnr_result["cnr_macro_f1"], 4)
)


In [ ]:

full_cnr_result = evaluate_cnr(
    resnet18_full,
    base / "models/resnet18_full_best.pth",
    "resnet18_full"
)

print("\nResNet18 Full External Evaluation Completed")
print(
    "Accuracy:",
    round(full_cnr_result["cnr_accuracy"] * 100, 2),
    "%"
)
print(
    "Macro F1:",
    round(full_cnr_result["cnr_macro_f1"], 4)
)


In [ ]:

import pandas as pd

cnr_results = [
    cnn_cnr_result,
    frozen_cnr_result,
    full_cnr_result
]

cnr_comparison = pd.DataFrame(cnr_results)

cnr_comparison["pklot_accuracy"] = [
    0.9310,
    0.96125,
    0.9905
]

cnr_comparison["accuracy_drop_pp"] = (
    cnr_comparison["pklot_accuracy"]
    - cnr_comparison["cnr_accuracy"]
) * 100

output_path = (
    base / "results/cnrpark_external_comparison.csv"
)

cnr_comparison.to_csv(output_path, index=False)

print(cnr_comparison.to_string(index=False))
print("\nSaved to:", output_path)


In [ ]:

import zipfile
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import shutil

zip_path = base / "data/raw/CNR-EXT-Patches-150x150.zip"
output_dir = Path("/content/cnrpark_full_test")

records = []

with zipfile.ZipFile(zip_path, "r") as z:
    label_lines = z.read("LABELS/test.txt").decode(
        "utf-8"
    ).splitlines()

    for line in label_lines:
        parts = line.strip().split()

        if len(parts) != 2:
            continue

        image_path, label = parts

        records.append({
            "image_path": image_path,
            "label": int(label),
            "class_name": (
                "empty" if int(label) == 0
                else "occupied"
            )
        })

    df = pd.DataFrame(records)

    print("Official test images:", len(df))
    print("\nClass distribution:")
    print(df["class_name"].value_counts())

    assert len(df) == 31825
    assert df["image_path"].is_unique
    assert set(df["label"].unique()) == {0, 1}

    for row in tqdm(
        df.itertuples(index=False),
        total=len(df),
        desc="Extracting official CNRPark test"
    ):
        # Preserve path information to prevent filename collisions
        unique_name = row.image_path.replace("/", "__")

        destination = (
            output_dir / row.class_name / unique_name
        )

        destination.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        source_path = "PATCHES/" + row.image_path

        with z.open(source_path) as src:
            with open(destination, "wb") as dst:
                shutil.copyfileobj(src, dst)

# Verify extracted image counts
empty_count = len(list(
    (output_dir / "empty").glob("*.jpg")
))

occupied_count = len(list(
    (output_dir / "occupied").glob("*.jpg")
))

print("\nExtraction completed!")
print("Empty:", empty_count)
print("Occupied:", occupied_count)
print("Total:", empty_count + occupied_count)

assert empty_count == 13549
assert occupied_count == 18276

# Save official test index
index_path = (
    base / "data/processed/cnrpark_official_test.csv"
)

df.to_csv(index_path, index=False)

print("Index saved:", index_path)


In [ ]:

import torch
from pathlib import Path
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Official CNRPark+EXT test directory
cnr_full_dir = Path("/content/cnrpark_full_test")

# Same preprocessing used for PKLot and CNRPark sample
cnr_full_transform = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

cnr_full_dataset = datasets.ImageFolder(
    root=cnr_full_dir,
    transform=cnr_full_transform
)

cnr_full_loader = DataLoader(
    cnr_full_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

print("Dataset size:", len(cnr_full_dataset))
print("Classes:", cnr_full_dataset.classes)
print("Class mapping:", cnr_full_dataset.class_to_idx)
print("Number of batches:", len(cnr_full_loader))

images, labels = next(iter(cnr_full_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

assert len(cnr_full_dataset) == 31825
assert cnr_full_dataset.class_to_idx == {
    "empty": 0,
    "occupied": 1
}

print("\nOfficial CNRPark DataLoader: READY")


In [ ]:

import torch
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix
)

def evaluate_cnr_official(model, model_name):
    model = model.to(device)
    model.eval()

    all_labels = []
    all_predictions = []

    with torch.inference_mode():
        for images, labels in tqdm(
            cnr_full_loader,
            desc=f"Evaluating {model_name}"
        ):
            images = images.to(device, non_blocking=True)

            outputs = model(images)
            predictions = outputs.argmax(dim=1)

            all_labels.extend(labels.tolist())
            all_predictions.extend(
                predictions.cpu().tolist()
            )

    accuracy = accuracy_score(
        all_labels, all_predictions
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            all_labels,
            all_predictions,
            average="macro",
            zero_division=0
        )
    )

    cm = confusion_matrix(
        all_labels,
        all_predictions,
        labels=[0, 1]
    )

    print(f"\n{model_name} - Official CNRPark+EXT Test")
    print("Images evaluated:", len(all_labels))
    print("Accuracy:", round(accuracy * 100, 2), "%")
    print("Macro Precision:", round(precision, 4))
    print("Macro Recall:", round(recall, 4))
    print("Macro F1:", round(f1, 4))
    print("Confusion Matrix:\n", cm)

    return {
        "model": model_name,
        "test_images": len(all_labels),
        "accuracy": accuracy,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
        "confusion_matrix": cm.tolist()
    }


# Evaluate CNN on all 31,825 official test images
cnn_official_result = evaluate_cnr_official(
    cnn_model,
    "cnn_from_scratch"
)


In [ ]:

# Load the saved Frozen ResNet18 checkpoint
checkpoint = torch.load(
    base / "models/resnet18_frozen_best.pth",
    map_location=device,
    weights_only=True
)

resnet18.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

# Evaluate on all 31,825 official CNRPark+EXT images
frozen_official_result = evaluate_cnr_official(
    resnet18,
    "resnet18_frozen"
)

print("\nFrozen ResNet18 official evaluation completed.")


In [ ]:

# Load the saved Full Fine-tuning checkpoint
checkpoint = torch.load(
    base / "models/resnet18_full_best.pth",
    map_location=device,
    weights_only=True
)

resnet18_full.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

# Evaluate on the official CNRPark+EXT test set
full_official_result = evaluate_cnr_official(
    resnet18_full,
    "resnet18_full"
)

print("\nFull Fine-tuning official evaluation completed.")


In [ ]:

import pandas as pd
import json

# Collect official evaluation results
official_results = [
    cnn_official_result,
    frozen_official_result,
    full_official_result
]

# Build comparison table
rows = []

pklot_accuracies = {
    "cnn_from_scratch": 0.9310,
    "resnet18_frozen": 0.96125,
    "resnet18_full": 0.9905
}

for result in official_results:
    name = result["model"]

    rows.append({
        "model": name,
        "dataset": "CNRPark+EXT",
        "split": "official_test",
        "test_images": result["test_images"],
        "input_resolution": "96x96",
        "decision_rule": "argmax",
        "accuracy": result["accuracy"],
        "macro_precision": result["macro_precision"],
        "macro_recall": result["macro_recall"],
        "macro_f1": result["macro_f1"],
        "pklot_test_accuracy": pklot_accuracies[name],
        "accuracy_drop_pp": (
            pklot_accuracies[name] - result["accuracy"]
        ) * 100
    })

official_df = pd.DataFrame(rows)

results_dir = base / "results"
results_dir.mkdir(parents=True, exist_ok=True)

csv_path = (
    results_dir / "cnrpark_official_test_comparison.csv"
)

official_df.to_csv(csv_path, index=False)

# Save confusion matrices separately
cm_path = (
    results_dir / "cnrpark_official_confusion_matrices.json"
)

confusion_matrices = {
    result["model"]: result["confusion_matrix"]
    for result in official_results
}

with open(cm_path, "w") as f:
    json.dump(confusion_matrices, f, indent=2)

print("Official CNRPark+EXT Results:")
print(official_df.to_string(index=False))

print("\nSaved files:")
print(csv_path)
print(cm_path)



In [ ]:

import torch
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm

# Use the best model
resnet18_full.eval()

wrong_examples = []
class_names = cnr_full_dataset.classes

with torch.inference_mode():
    for images, labels, in tqdm(
        cnr_full_loader,
        desc="Collecting classification errors"
    ):
        images = images.to(device)
        predictions = resnet18_full(images).argmax(dim=1)

        for image, true_label, pred_label in zip(
            images.cpu(),
            labels,
            predictions.cpu()
        ):
            if true_label.item() != pred_label.item():
                wrong_examples.append({
                    "image": image,
                    "true": true_label.item(),
                    "pred": pred_label.item()
                })

        if len(wrong_examples) >= 12:
            break

# Reverse ImageNet normalization for visualization
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

fig, axes = plt.subplots(3, 4, figsize=(13, 10))

for ax, example in zip(axes.flat, wrong_examples[:12]):
    image = example["image"] * std + mean
    image = image.clamp(0, 1).permute(1, 2, 0).numpy()

    ax.imshow(image)
    ax.set_title(
        f"True: {class_names[example['true']]}\n"
        f"Pred: {class_names[example['pred']]}",
        fontsize=10
    )
    ax.axis("off")

for ax in axes.flat[len(wrong_examples[:12]):]:
    ax.axis("off")

plt.suptitle(
    "ResNet18 Full - CNRPark+EXT Error Gallery",
    fontsize=15
)
plt.tight_layout()

output_path = (
    base / "results/resnet18_full_cnr_error_gallery.png"
)

plt.savefig(output_path, dpi=150, bbox_inches="tight")
plt.show()

print("Errors displayed:", min(len(wrong_examples), 12))
print("Saved to:", output_path)


In [ ]:

import torch
import matplotlib.pyplot as plt
import random

# Reproducible selection
random.seed(42)

resnet18_full.eval()

error_groups = {
    "empty_to_occupied": [],
    "occupied_to_empty": []
}

with torch.inference_mode():
    for images, labels in cnr_full_loader:
        images_gpu = images.to(device)

        predictions = (
            resnet18_full(images_gpu)
            .argmax(dim=1)
            .cpu()
        )

        for image, true_label, pred_label in zip(
            images,
            labels,
            predictions
        ):
            true_label = true_label.item()
            pred_label = pred_label.item()

            if true_label == 0 and pred_label == 1:
                error_groups["empty_to_occupied"].append(
                    image.clone()
                )

            elif true_label == 1 and pred_label == 0:
                error_groups["occupied_to_empty"].append(
                    image.clone()
                )

print("Empty → Occupied:",
      len(error_groups["empty_to_occupied"]))

print("Occupied → Empty:",
      len(error_groups["occupied_to_empty"]))

assert len(error_groups["empty_to_occupied"]) == 105
assert len(error_groups["occupied_to_empty"]) == 1989

mean = torch.tensor(
    [0.485, 0.456, 0.406]
).view(3, 1, 1)

std = torch.tensor(
    [0.229, 0.224, 0.225]
).view(3, 1, 1)

fig, axes = plt.subplots(2, 6, figsize=(16, 6))

for row, (error_type, images) in enumerate(
    error_groups.items()
):
    selected = random.sample(images, 6)

    for col, image in enumerate(selected):
        image = (image * std + mean).clamp(0, 1)
        image = image.permute(1, 2, 0).numpy()

        axes[row, col].imshow(image)
        axes[row, col].axis("off")

        axes[row, col].set_title(
            error_type.replace("_", " → "),
            fontsize=9
        )

plt.suptitle(
    "ResNet18 Full - Balanced Error Analysis",
    fontsize=15
)

plt.tight_layout()

output_path = (
    base / "results/"
    "resnet18_full_balanced_error_gallery.png"
)

plt.savefig(
    output_path,
    dpi=150,
    bbox_inches="tight"
)

plt.show()

print("\nSaved to:", output_path)


In [ ]:

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

resnet18_full.eval()

all_confidences = []
all_correct = []

with torch.inference_mode():
    for images, labels in tqdm(
        cnr_full_loader,
        desc="Computing ECE"
    ):
        images = images.to(device)

        outputs = resnet18_full(images)
        probabilities = torch.softmax(outputs, dim=1)

        confidences, predictions = probabilities.max(dim=1)

        all_confidences.extend(
            confidences.cpu().numpy()
        )

        all_correct.extend(
            (predictions.cpu() == labels).numpy()
        )

all_confidences = np.array(all_confidences)
all_correct = np.array(all_correct, dtype=float)

# Expected Calibration Error with 15 bins
n_bins = 15
bin_edges = np.linspace(0, 1, n_bins + 1)

ece = 0.0
bin_accuracies = []
bin_confidences = []
bin_counts = []

for i in range(n_bins):
    lower = bin_edges[i]
    upper = bin_edges[i + 1]

    if i == 0:
        mask = (
            (all_confidences >= lower)
            & (all_confidences <= upper)
        )
    else:
        mask = (
            (all_confidences > lower)
            & (all_confidences <= upper)
        )

    count = mask.sum()

    if count > 0:
        accuracy = all_correct[mask].mean()
        confidence = all_confidences[mask].mean()

        ece += (
            count / len(all_correct)
        ) * abs(accuracy - confidence)
    else:
        accuracy = np.nan
        confidence = np.nan

    bin_accuracies.append(accuracy)
    bin_confidences.append(confidence)
    bin_counts.append(count)

print("\nResNet18 Full - Official CNRPark+EXT")
print("Images:", len(all_correct))
print("Accuracy:", round(all_correct.mean() * 100, 2), "%")
print("ECE (15 bins):", round(ece, 4))

# Save calibration data
calibration_df = pd.DataFrame({
    "bin_lower": bin_edges[:-1],
    "bin_upper": bin_edges[1:],
    "accuracy": bin_accuracies,
    "confidence": bin_confidences,
    "count": bin_counts
})

calibration_df.to_csv(
    base / "results/resnet18_full_cnr_calibration.csv",
    index=False
)

# Reliability diagram
valid = np.array(bin_counts) > 0

plt.figure(figsize=(7, 6))

plt.plot(
    [0, 1], [0, 1],
    "--",
    label="Perfect calibration"
)

plt.plot(
    np.array(bin_confidences)[valid],
    np.array(bin_accuracies)[valid],
    "o-",
    label="ResNet18 Full"
)

plt.xlabel("Mean Confidence")
plt.ylabel("Observed Accuracy")
plt.title(
    f"CNRPark+EXT Reliability Diagram (ECE={ece:.4f})"
)
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.legend()
plt.grid(alpha=0.3)

plt.savefig(
    base / "results/resnet18_full_cnr_reliability.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()


In [ ]:

import pandas as pd
from statsmodels.stats.proportion import proportion_confint

# Official CNRPark+EXT confusion matrices
models_cm = {
    "cnn_from_scratch": [[13027, 522], [8656, 9620]],
    "resnet18_frozen": [[13374, 175], [3179, 15097]],
    "resnet18_full": [[13444, 105], [1989, 16287]]
}

ci_results = []

for model_name, cm in models_cm.items():
    correct = cm[0][0] + cm[1][1]
    total = sum(sum(row) for row in cm)

    ci_low, ci_high = proportion_confint(
        count=correct,
        nobs=total,
        alpha=0.05,
        method="wilson"
    )

    ci_results.append({
        "model": model_name,
        "test_images": total,
        "accuracy": correct / total,
        "ci_95_lower": ci_low,
        "ci_95_upper": ci_high
    })

ci_df = pd.DataFrame(ci_results)

output_path = (
    base / "results/"
    "cnrpark_official_accuracy_confidence_intervals.csv"
)

ci_df.to_csv(output_path, index=False)

print("Official CNRPark+EXT - 95% Confidence Intervals\n")

for row in ci_results:
    print(row["model"])
    print(f"  Accuracy: {row['accuracy']:.2%}")
    print(
        f"  95% CI: "
        f"[{row['ci_95_lower']:.2%}, "
        f"{row['ci_95_upper']:.2%}]"
    )
    print()

print("Saved to:", output_path)


In [ ]:

from pathlib import Path

print("PROJECT FILE AUDIT")
print("=" * 60)

folders = [
    "data/processed",
    "data/processed/splits",
    "data/processed/samples",
    "models",
    "results"
]

for folder in folders:
    path = base / folder

    print(f"\n[{folder}]")

    if not path.exists():
        print("MISSING FOLDER")
        continue

    files = sorted(
        p for p in path.iterdir()
        if p.is_file()
    )

    if files:
        for file in files:
            print(" -", file.name)
    else:
        print("No files at this level")

print("\nROOT FILES")
print("=" * 60)

for file in sorted(base.iterdir()):
    if file.is_file():
        print(" -", file.name)


In [ ]:

import pandas as pd

sample_dir = base / "data/processed/samples"

files = {
    "Train": sample_dir / "train_clean.csv",
    "Validation": sample_dir / "val_clean.csv",
    "Test": sample_dir / "test_clean.csv"
}

for split_name, file_path in files.items():
    df = pd.read_csv(file_path)

    print("\n" + "=" * 55)
    print(f"{split_name.upper()} DATA CARD")
    print("=" * 55)

    print("Total images:", len(df))
    print("Columns:", df.columns.tolist())

    print("\nMissing values:")
    print(df.isnull().sum().to_string())

    print("\nDuplicate rows:", df.duplicated().sum())

    # Detect useful metadata columns
    keywords = [
        "label", "class", "site", "parking",
        "camera", "date", "day", "weather"
    ]

    for col in df.columns:
        if any(word in col.lower() for word in keywords):
            print(f"\n{col}:")
            counts = df[col].value_counts(dropna=False)

            if len(counts) <= 15:
                print(counts.to_string())
            else:
                print("Unique values:", df[col].nunique())
                print(counts.head(5).to_string())

print("\nDATA CARD AUDIT COMPLETED")


In [ ]:

import pandas as pd

sample_dir = base / "data/processed/samples"

splits = {
    "Train": pd.read_csv(sample_dir / "train_clean.csv"),
    "Validation": pd.read_csv(sample_dir / "val_clean.csv"),
    "Test": pd.read_csv(sample_dir / "test_clean.csv")
}

print("DATA LEAKAGE AUDIT")
print("=" * 60)

for name, df in splits.items():
    print(f"\n{name}")
    print("Locations:", df["location"].unique().tolist())
    print("Unique dates:", df["date"].nunique())
    print("Unique image paths:", df["image_path"].nunique())

pairs = [
    ("Train", "Validation"),
    ("Train", "Test"),
    ("Validation", "Test")
]

for name_a, name_b in pairs:
    a = splits[name_a]
    b = splits[name_b]

    shared_paths = (
        set(a["image_path"]) & set(b["image_path"])
    )

    shared_locations = (
        set(a["location"]) & set(b["location"])
    )

    shared_dates = (
        set(a["date"]) & set(b["date"])
    )

    # More relevant for same-site temporal leakage
    a_location_dates = set(
        zip(a["location"], a["date"])
    )
    b_location_dates = set(
        zip(b["location"], b["date"])
    )

    shared_location_dates = (
        a_location_dates & b_location_dates
    )

    print("\n" + "-" * 60)
    print(f"{name_a} vs {name_b}")
    print("Shared image paths:", len(shared_paths))
    print("Shared locations:", sorted(shared_locations))
    print("Shared dates:", len(shared_dates))
    print(
        "Shared location-date combinations:",
        len(shared_location_dates)
    )

print("\nDATA LEAKAGE AUDIT COMPLETED")



In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

# Load cleaned training metadata
train_df = pd.read_csv(
    base / "data/processed/samples/train_clean.csv"
)

# Reproducible balanced sample: 50 per class
audit_df = (
    train_df.groupby("label", group_keys=False)
    .sample(n=50, random_state=42)
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

audit_df.insert(0, "audit_id", range(1, 101))

# Save audit index
audit_path = (
    base / "results/pklot_label_audit_100.csv"
)

audit_df["reviewed_label"] = ""
audit_df["label_correct"] = ""
audit_df["notes"] = ""

audit_df.to_csv(audit_path, index=False)

print("Audit images:", len(audit_df))
print("\nClass distribution:")
print(audit_df["label"].value_counts())
print("\nSaved audit index:", audit_path)

# Display the first 20 entries to inspect metadata
print("\nFirst 20 audit entries:")
print(
    audit_df[
        ["audit_id", "label", "weather", "date"]
    ].head(20).to_string(index=False)
)


In [ ]:

import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

# Display first 20 images from the label audit
batch = audit_df.iloc[0:20]

fig, axes = plt.subplots(4, 5, figsize=(15, 12))

for ax, (_, row) in zip(axes.flat, batch.iterrows()):
    # Use the extracted, cleaned training images
    image_path = (
        base / "data/processed/images_clean/train"
        / row["label"]
        / Path(row["image_path"]).name
    )

    if image_path.exists():
        with Image.open(image_path) as img:
            ax.imshow(img.convert("RGB"))
    else:
        ax.text(
            0.5, 0.5,
            "IMAGE NOT FOUND",
            ha="center",
            va="center"
        )

    ax.set_title(
        f"ID {row['audit_id']} | {row['label']}",
        fontsize=10
    )
    ax.axis("off")

plt.suptitle(
    "PKLot Label Audit - Images 1 to 20",
    fontsize=16
)

plt.tight_layout()

output_path = (
    base / "results/pklot_label_audit_batch_01.png"
)

plt.savefig(
    output_path,
    dpi=150,
    bbox_inches="tight"
)

plt.show()

print("Audit batch saved:", output_path)


In [ ]:

from pathlib import Path

print("IMAGE PATH DIAGNOSTIC")
print("=" * 60)

# Original path stored in the CSV
row = audit_df.iloc[0]

print("CSV image_path:")
print(row["image_path"])

print("\nCSV label:")
print(row["label"])

print("\nExpected filename:")
print(Path(row["image_path"]).name)

# Check Drive image directory
drive_dir = base / "data/processed/images_clean/train"

print("\nDrive directory exists:", drive_dir.exists())

for label in ["empty", "occupied"]:
    folder = drive_dir / label
    print(f"\nDrive {label} exists:", folder.exists())

    if folder.exists():
        files = list(folder.iterdir())
        print("Number of files:", len(files))
        print("First 3 filenames:")
        for p in files[:3]:
            print(" -", p.name)

# Check local Colab image directory
local_dir = Path("/content/parking_images/train")

print("\nLocal directory exists:", local_dir.exists())

for label in ["empty", "occupied"]:
    folder = local_dir / label
    print(f"\nLocal {label} exists:", folder.exists())

    if folder.exists():
        files = list(folder.iterdir())
        print("Number of files:", len(files))
        print("First 3 filenames:")
        for p in files[:3]:
            print(" -", p.name)

print("\nDIAGNOSTIC COMPLETED")



In [ ]:

import hashlib
from pathlib import Path

row = audit_df.iloc[0]
original_path = str(row["image_path"])
original_name = Path(original_path).name

folder = base / "data/processed/images_clean/train" / row["label"]

candidates = {
    "MD5 full path": hashlib.md5(original_path.encode()).hexdigest(),
    "MD5 filename": hashlib.md5(original_name.encode()).hexdigest(),
    "SHA1 full path": hashlib.sha1(original_path.encode()).hexdigest(),
    "SHA1 filename": hashlib.sha1(original_name.encode()).hexdigest(),
    "SHA256 full path": hashlib.sha256(original_path.encode()).hexdigest(),
}

print("Original path:", original_path)
print("\nChecking possible filename patterns:")
print("=" * 60)

matches = []

for method, value in candidates.items():
    for length in [16, 20, 24, 32, 40, 64]:
        candidate = folder / f"{value[:length]}.jpg"

        if candidate.exists():
            matches.append((method, length, candidate))
            print("MATCH:", method, "length:", length)
            print("File:", candidate.name)

if not matches:
    print("No matching hash pattern found.")

print("\nFirst existing filename:")
print(next(folder.iterdir()).name)

print("\nHASH DIAGNOSTIC COMPLETED")


In [ ]:

import hashlib
import matplotlib.pyplot as plt
from PIL import Image

def get_audit_image_path(row):
    original_path = str(row["image_path"])

    hashed_name = (
        hashlib.sha256(original_path.encode())
        .hexdigest()[:20] + ".jpg"
    )

    return (
        base / "data/processed/images_clean/train"
        / row["label"]
        / hashed_name
    )

batch = audit_df.iloc[:20]

fig, axes = plt.subplots(4, 5, figsize=(15, 12))
missing = []

for ax, (_, row) in zip(axes.flat, batch.iterrows()):
    image_path = get_audit_image_path(row)

    if image_path.exists():
        with Image.open(image_path) as img:
            ax.imshow(img.convert("RGB"))
    else:
        missing.append(int(row["audit_id"]))
        ax.text(
            0.5, 0.5, "IMAGE NOT FOUND",
            ha="center", va="center"
        )

    ax.set_title(
        f"ID {row['audit_id']} | {row['label']}",
        fontsize=10
    )
    ax.axis("off")

plt.suptitle(
    "PKLot Label Audit - Images 1 to 20",
    fontsize=16
)
plt.tight_layout()

output_path = (
    base / "results/pklot_label_audit_batch_01.png"
)

plt.savefig(output_path, dpi=150, bbox_inches="tight")
plt.show()

print("Images displayed:", 20 - len(missing))
print("Missing image IDs:", missing)
print("Saved to:", output_path)


In [ ]:

import matplotlib.pyplot as plt
from PIL import Image

batch = audit_df.iloc[20:40]

fig, axes = plt.subplots(4, 5, figsize=(15, 12))
missing = []

for ax, (_, row) in zip(axes.flat, batch.iterrows()):
    image_path = get_audit_image_path(row)

    if image_path.exists():
        with Image.open(image_path) as img:
            ax.imshow(img.convert("RGB"))
    else:
        missing.append(int(row["audit_id"]))
        ax.text(
            0.5, 0.5, "IMAGE NOT FOUND",
            ha="center", va="center"
        )

    ax.set_title(
        f"ID {row['audit_id']} | {row['label']}",
        fontsize=10
    )
    ax.axis("off")

plt.suptitle(
    "PKLot Label Audit - Images 21 to 40",
    fontsize=16
)

plt.tight_layout()

output_path = (
    base / "results/pklot_label_audit_batch_02.png"
)

plt.savefig(output_path, dpi=150, bbox_inches="tight")
plt.show()

print("Images displayed:", 20 - len(missing))
print("Missing image IDs:", missing)
print("Saved to:", output_path)


In [ ]:

import matplotlib.pyplot as plt
from PIL import Image

batch = audit_df.iloc[40:60]

fig, axes = plt.subplots(4, 5, figsize=(15, 12))
missing = []

for ax, (_, row) in zip(axes.flat, batch.iterrows()):
    image_path = get_audit_image_path(row)

    if image_path.exists():
        with Image.open(image_path) as img:
            ax.imshow(img.convert("RGB"))
    else:
        missing.append(int(row["audit_id"]))
        ax.text(
            0.5, 0.5, "IMAGE NOT FOUND",
            ha="center", va="center"
        )

    ax.set_title(
        f"ID {row['audit_id']} | {row['label']}",
        fontsize=10
    )
    ax.axis("off")

plt.suptitle(
    "PKLot Label Audit - Images 41 to 60",
    fontsize=16
)

plt.tight_layout()

output_path = (
    base / "results/pklot_label_audit_batch_03.png"
)

plt.savefig(output_path, dpi=150, bbox_inches="tight")
plt.show()

print("Images displayed:", 20 - len(missing))
print("Missing image IDs:", missing)
print("Saved to:", output_path)


In [ ]:

import matplotlib.pyplot as plt
from PIL import Image

batch = audit_df.iloc[60:80]

fig, axes = plt.subplots(4, 5, figsize=(15, 12))
missing = []

for ax, (_, row) in zip(axes.flat, batch.iterrows()):
    image_path = get_audit_image_path(row)

    if image_path.exists():
        with Image.open(image_path) as img:
            ax.imshow(img.convert("RGB"))
    else:
        missing.append(int(row["audit_id"]))
        ax.text(
            0.5, 0.5, "IMAGE NOT FOUND",
            ha="center", va="center"
        )

    ax.set_title(
        f"ID {row['audit_id']} | {row['label']}",
        fontsize=10
    )
    ax.axis("off")

plt.suptitle(
    "PKLot Label Audit - Images 61 to 80",
    fontsize=16
)

plt.tight_layout()

output_path = (
    base / "results/pklot_label_audit_batch_04.png"
)

plt.savefig(output_path, dpi=150, bbox_inches="tight")
plt.show()

print("Images displayed:", 20 - len(missing))
print("Missing image IDs:", missing)
print("Saved to:", output_path)


In [ ]:

import matplotlib.pyplot as plt
from PIL import Image

batch = audit_df.iloc[80:100]

fig, axes = plt.subplots(4, 5, figsize=(15, 12))
missing = []

for ax, (_, row) in zip(axes.flat, batch.iterrows()):
    image_path = get_audit_image_path(row)

    if image_path.exists():
        with Image.open(image_path) as img:
            ax.imshow(img.convert("RGB"))
    else:
        missing.append(int(row["audit_id"]))
        ax.text(
            0.5, 0.5, "IMAGE NOT FOUND",
            ha="center", va="center"
        )

    ax.set_title(
        f"ID {row['audit_id']} | {row['label']}",
        fontsize=10
    )
    ax.axis("off")

plt.suptitle(
    "PKLot Label Audit - Images 81 to 100",
    fontsize=16
)

plt.tight_layout()

output_path = (
    base / "results/pklot_label_audit_batch_05.png"
)

plt.savefig(output_path, dpi=150, bbox_inches="tight")
plt.show()

print("Images displayed:", 20 - len(missing))
print("Missing image IDs:", missing)
print("Saved to:", output_path)


In [ ]:

import pandas as pd
from scipy.stats import beta

# Visual audit results
n_reviewed = 100
n_errors = 0
confidence = 0.95

observed_error_rate = n_errors / n_reviewed

# One-sided 95% upper confidence bound
if n_errors == n_reviewed:
    upper_bound = 1.0
else:
    upper_bound = beta.ppf(
        confidence,
        n_errors + 1,
        n_reviewed - n_errors
    )

# Update audit file
audit_file = base / "results/pklot_label_audit_100.csv"
audit_results = pd.read_csv(audit_file)

audit_results["reviewed_label"] = audit_results["label"]
audit_results["label_correct"] = True
audit_results["notes"] = (
    "No obvious label mismatch in visual review"
)

audit_results.to_csv(audit_file, index=False)

# Save summary
summary = pd.DataFrame([{
    "dataset": "PKLot",
    "split": "train",
    "review_method": "visual inspection",
    "sample_size": n_reviewed,
    "empty_samples": 50,
    "occupied_samples": 50,
    "observed_label_errors": n_errors,
    "observed_error_rate": observed_error_rate,
    "confidence_level": confidence,
    "upper_error_rate_bound": upper_bound,
    "interval_method": "Clopper-Pearson one-sided"
}])

summary_path = (
    base / "results/pklot_label_audit_summary.csv"
)

summary.to_csv(summary_path, index=False)

print("PKLOT LABEL AUDIT SUMMARY")
print("=" * 55)
print("Reviewed images:", n_reviewed)
print("Observed label errors:", n_errors)
print(f"Observed error rate: {observed_error_rate:.2%}")
print(f"95% upper confidence bound: {upper_bound:.2%}")
print("\nSaved audit:", audit_file)
print("Saved summary:", summary_path)


In [ ]:

from pathlib import Path
import pandas as pd

splits = {
    "Train": pd.read_csv(
        base / "data/processed/samples/train_clean.csv"
    ),
    "Validation": pd.read_csv(
        base / "data/processed/samples/val_clean.csv"
    ),
    "Test": pd.read_csv(
        base / "data/processed/samples/test_clean.csv"
    )
}

audit = pd.read_csv(
    base / "results/pklot_label_audit_summary.csv"
).iloc[0]

lines = [
    "# PKLot Dataset Card",
    "",
    "## 1. Dataset Overview",
    "",
    "- Dataset: PKLot",
    "- Task: Binary parking-space occupancy classification",
    "- Classes: empty, occupied",
    "- Source: PKLot parking-space image dataset",
    "- Input resolution used for modeling: 96 x 96 RGB",
    "",
    "## 2. Cleaned Sample Statistics",
    "",
    "| Split | Images | Empty | Occupied | Locations | Unique Days |",
    "|---|---:|---:|---:|---|---:|"
]

for name, df in splits.items():
    counts = df["label"].value_counts()
    locations = ", ".join(sorted(df["location"].unique()))

    lines.append(
        f"| {name} | {len(df)} | "
        f"{counts.get('empty', 0)} | "
        f"{counts.get('occupied', 0)} | "
        f"{locations} | {df['date'].nunique()} |"
    )

lines += [
    "",
    "## 3. Weather Distribution",
    "",
    "| Split | Sunny | Cloudy | Rainy |",
    "|---|---:|---:|---:|"
]

for name, df in splits.items():
    counts = df["weather"].value_counts()

    lines.append(
        f"| {name} | "
        f"{counts.get('Sunny', 0)} | "
        f"{counts.get('Cloudy', 0)} | "
        f"{counts.get('Rainy', 0)} |"
    )

lines += [
    "",
    "## 4. Data Preparation",
    "",
    "- Cleaned and balanced samples were prepared for each split.",
    "- No missing metadata values were detected.",
    "- No duplicate metadata rows were detected within each split.",
    "- Images were resized to 96 x 96 pixels.",
    "- ImageNet normalization was applied.",
    "- Random horizontal flipping was used for training augmentation.",
    "",
    "## 5. Split Strategy and Leakage Prevention",
    "",
    "- Training: PUC, 29 days.",
    "- Validation: PUC, 9 different days.",
    "- Test: UFPR04 and UFPR05, 63 days.",
    "- No overlapping image paths across splits.",
    "- No overlapping location-date combinations across splits.",
    "- Training and validation share a site but use disjoint dates.",
    "- Test locations are disjoint from training and validation.",
    "- Content-based duplicate screening was also performed "
    "during dataset preparation.",
    "",
    "## 6. Label Quality Audit",
    "",
    "- Method: Visual review of 100 sampled training images.",
    "- Sampling: 50 empty and 50 occupied images.",
    f"- Observed label errors: {int(audit['observed_label_errors'])}.",
    f"- Observed error rate: {audit['observed_error_rate']:.2%}.",
    f"- One-sided 95% upper error-rate bound: "
    f"{audit['upper_error_rate_bound']:.2%}.",
    "- Confidence method: Clopper-Pearson.",
    "- The bound assumes independent samples and reliable visual review.",
    "- No obvious label mismatches were observed.",
    "",
    "## 7. Limitations",
    "",
    "- Weather distributions differ between splits.",
    "- Images from the same cameras may be correlated.",
    "- Label quality was assessed on a sample, not the full dataset.",
    "- The test set evaluates transfer to different PKLot locations.",
    "- CNRPark+EXT is used separately for external domain-shift evaluation.",
    "",
    "## 8. Intended Use",
    "",
    "- Research and educational evaluation of parking occupancy classifiers.",
    "- Results should not be interpreted as guaranteed performance "
    "across all cameras, lighting conditions, or parking facilities.",
]

output_path = base / "PKLot_Data_Card.md"

output_path.write_text(
    "\n".join(lines) + "\n",
    encoding="utf-8"
)

print("DATA CARD CREATED")
print("=" * 55)
print("Saved to:", output_path)
print("File size:", output_path.stat().st_size, "bytes")
print("\nPreview:")
print("\n".join(lines[:25]))


In [ ]:

import sys
import platform
import importlib.metadata as metadata

print("PROJECT ENVIRONMENT AUDIT")
print("=" * 60)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

packages = [
    "torch",
    "torchvision",
    "numpy",
    "pandas",
    "scikit-learn",
    "scipy",
    "matplotlib",
    "Pillow",
    "tqdm",
    "wandb",
    "statsmodels"
]

print("\nInstalled package versions:")
for package in packages:
    try:
        version = metadata.version(package)
        print(f"{package}=={version}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

print("\nPyTorch hardware:")
import torch

print("CUDA available:", torch.cuda.is_available())
print("PyTorch CUDA version:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nENVIRONMENT AUDIT COMPLETED")


In [ ]:

from pathlib import Path
import importlib.metadata as metadata
import sys
import platform
import torch

packages = [
    "torch",
    "torchvision",
    "numpy",
    "pandas",
    "scikit-learn",
    "scipy",
    "matplotlib",
    "Pillow",
    "tqdm",
    "wandb",
    "statsmodels"
]

# Save exact installed versions
requirements = []

for package in packages:
    version = metadata.version(package)
    requirements.append(f"{package}=={version}")

requirements_path = base / "requirements.txt"

requirements_path.write_text(
    "\n".join(requirements) + "\n",
    encoding="utf-8"
)

# Save runtime environment metadata
environment = [
    "# Project Runtime Environment",
    "",
    f"Python: {sys.version.split()[0]}",
    f"Platform: {platform.platform()}",
    f"PyTorch: {torch.__version__}",
    f"CUDA: {torch.version.cuda}",
    f"CUDA available: {torch.cuda.is_available()}",
]

if torch.cuda.is_available():
    environment.append(
        f"GPU: {torch.cuda.get_device_name(0)}"
    )

environment_path = base / "environment_info.txt"

environment_path.write_text(
    "\n".join(environment) + "\n",
    encoding="utf-8"
)

print("REPRODUCIBILITY FILES CREATED")
print("=" * 55)

print("\nrequirements.txt:")
print(requirements_path.read_text())

print("environment_info.txt:")
print(environment_path.read_text())

print("Files saved successfully.")


In [ ]:

import pandas as pd

history_files = {
    "CNN From Scratch": "cnn_training_history.csv",
    "ResNet18 Frozen": "resnet18_frozen_history.csv",
    "ResNet18 Full": "resnet18_full_history.csv"
}

print("TRAINING CONFIGURATION AUDIT")
print("=" * 60)

for model_name, filename in history_files.items():
    path = base / "results" / filename

    print(f"\nMODEL: {model_name}")
    print("-" * 60)

    if not path.exists():
        print("FILE NOT FOUND:", filename)
        continue

    df = pd.read_csv(path)

    print("History file:", filename)
    print("Columns:", df.columns.tolist())
    print("Recorded epochs:", len(df))

    print("\nFirst 3 rows:")
    print(df.head(3).to_string(index=False))

    print("\nLast 3 rows:")
    print(df.tail(3).to_string(index=False))

print("\nTRAINING CONFIGURATION AUDIT COMPLETED")


In [ ]:

from google.colab import drive
from pathlib import Path
import pandas as pd

# Connect Google Drive
drive.mount("/content/drive")

# Project directory
base = Path(
    "/content/drive/MyDrive/Traffic_Parking_Project"
)

print("Project found:", base.exists())

history_files = {
    "CNN From Scratch": "cnn_training_history.csv",
    "ResNet18 Frozen": "resnet18_frozen_history.csv",
    "ResNet18 Full": "resnet18_full_history.csv"
}

print("\nTRAINING CONFIGURATION AUDIT")
print("=" * 60)

for model_name, filename in history_files.items():

    path = base / "results" / filename

    print(f"\nMODEL: {model_name}")
    print("-" * 60)

    if not path.exists():
        print("FILE NOT FOUND:", filename)
        continue

    df = pd.read_csv(path)

    print("History file:", filename)
    print("Columns:", df.columns.tolist())
    print("Recorded epochs:", len(df))

    print("\nFirst 3 rows:")
    print(df.head(3).to_string(index=False))

    print("\nLast 3 rows:")
    print(df.tail(3).to_string(index=False))

print("\nTRAINING CONFIGURATION AUDIT COMPLETED")


In [ ]:

from pathlib import Path

print("TRAINING NOTEBOOK AUDIT")
print("=" * 60)

notebooks = sorted(base.rglob("*.ipynb"))

print("Notebooks found:", len(notebooks))

for notebook in notebooks:
    print(
        "-",
        notebook.relative_to(base)
    )

print("\nTRAINING NOTEBOOK AUDIT COMPLETED")


In [ ]:

from pathlib import Path

drive_root = Path("/content/drive/MyDrive")

print("GOOGLE DRIVE NOTEBOOK SEARCH")
print("=" * 60)

notebooks = sorted(drive_root.rglob("*.ipynb"))

print("Total notebooks found:", len(notebooks))

for i, notebook in enumerate(notebooks, 1):
    print(f"{i}. {notebook.relative_to(drive_root)}")

print("\nNOTEBOOK SEARCH COMPLETED")


In [ ]:

import json
from pathlib import Path

notebook_path = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "Traffic_Parking_Phase1.ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

keywords = [
    "manual_seed",
    "random.seed",
    "np.random.seed",
    "batch_size",
    "learning_rate",
    "lr=",
    "optim.",
    "optimizer",
    "scheduler",
    "CrossEntropyLoss",
    "num_epochs",
    "epochs =",
    "randomhorizontalflip",
    "randomresizedcrop",
    "weights=",
    "requires_grad",
    "best_val"
]

print("TRAINING SETTINGS EXTRACTION")
print("=" * 60)

matches = 0

for i, cell in enumerate(notebook.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    lines = source.splitlines()

    matching_lines = [
        (j + 1, line.strip())
        for j, line in enumerate(lines)
        if any(k in line.lower() for k in keywords)
    ]

    if matching_lines:
        print(f"\nCELL {i + 1}")

        for line_number, line in matching_lines:
            print(f"  Line {line_number}: {line[:180]}")
            matches += 1

print("\nTotal matching lines:", matches)
print("TRAINING SETTINGS EXTRACTION COMPLETED")


In [ ]:

import json

training_config = {
    "project": "Traffic Density and Parking Occupancy Analysis",
    "phase": "M1-M3",
    "dataset": "PKLot",
    "task": "Binary parking occupancy classification",
    "classes": ["empty", "occupied"],
    "input_size": [96, 96],
    "random_seed": 42,
    "batch_size": 32,
    "normalization": "ImageNet mean and std",
    "checkpoint_selection": "minimum validation loss",
    "models": {
        "cnn_from_scratch": {
            "epochs": 5,
            "optimizer": "Adam",
            "learning_rate": 0.001,
            "pretrained": False,
            "checkpoint": "models/parking_cnn_best.pth"
        },
        "resnet18_frozen": {
            "epochs": 5,
            "optimizer": "Adam",
            "learning_rate": 0.001,
            "pretrained": True,
            "trainable_layers": "final classification layer",
            "checkpoint": "models/resnet18_frozen_best.pth"
        },
        "resnet18_full": {
            "epochs": 5,
            "optimizer": "Adam",
            "learning_rate": 0.0001,
            "pretrained": True,
            "trainable_layers": "all layers",
            "checkpoint": "models/resnet18_full_best.pth"
        }
    },
    "external_evaluation": {
        "dataset": "CNRPark+EXT",
        "official_test_images": 31825,
        "input_size": [96, 96]
    }
}

output_path = base / "training_config.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        training_config,
        f,
        indent=4,
        ensure_ascii=False
    )

print("TRAINING CONFIG CREATED")
print("=" * 55)
print("Saved to:", output_path)
print("\nModel configurations:")

for name, config in training_config["models"].items():
    print(
        f"{name}: "
        f"epochs={config['epochs']}, "
        f"lr={config['learning_rate']}, "
        f"optimizer={config['optimizer']}"
    )

print("\nCONFIGURATION SAVED SUCCESSFULLY")


In [ ]:

from pathlib import Path

required_files = {
    "Documentation": [
        "PKLot_Data_Card.md",
        "requirements.txt",
        "environment_info.txt",
        "training_config.json"
    ],
    "Model Checkpoints": [
        "models/parking_cnn_best.pth",
        "models/resnet18_frozen_best.pth",
        "models/resnet18_full_best.pth"
    ],
    "Training Histories": [
        "results/cnn_training_history.csv",
        "results/resnet18_frozen_history.csv",
        "results/resnet18_full_history.csv"
    ],
    "Model Evaluation": [
        "results/final_model_comparison.csv",
        "results/cnrpark_official_test_comparison.csv",
        "results/cnrpark_official_accuracy_confidence_intervals.csv",
        "results/cnrpark_official_confusion_matrices.json"
    ],
    "Error Analysis and Calibration": [
        "results/resnet18_full_balanced_error_gallery.png",
        "results/resnet18_full_cnr_calibration.csv",
        "results/resnet18_full_cnr_reliability.png",
        "results/pklot_label_audit_summary.csv"
    ]
}

print("PHASE 1 DELIVERABLES AUDIT")
print("=" * 60)

total = 0
found = 0

for category, filenames in required_files.items():
    print(f"\n{category}")
    print("-" * 45)

    for filename in filenames:
        total += 1
        path = base / filename

        if path.is_file() and path.stat().st_size > 0:
            found += 1
            print("[OK]     ", filename)
        else:
            print("[MISSING]", filename)

print("\n" + "=" * 60)
print(f"Files found: {found}/{total}")
print(f"Files missing: {total - found}")
print("PHASE 1 DELIVERABLES AUDIT COMPLETED")
